# Methodology figures and model designs

This notebook generates dissertation-quality methodology schematics for the OpenBind ligand-only five-fold cross-validation study. The figures are based on the frozen dataset metadata and implemented model configurations; they do not contain performance results.

## Recommended figure set

The four essential main-text figures are: **(1)** the complete experimental workflow, **(2)** leakage-controlled cross-validation, **(3)** the controlled molecular-representation hierarchy and **(4)** the detailed adapted-MGT architecture. The curation/geometry audit and training/masking schematic provide valuable reproducibility detail and can remain in the main chapter if space permits, or move to an appendix. A separate Huber-loss curve is not recommended because the equation and short explanation are clearer than another figure.

## Code used and shared imports

I read the saved curation and CV metadata, then use a shared set of drawing helpers to keep the methodology figures consistent. These cells draw the study design; they do not prepare data or train models.

All imports are collected below: `Path` and `json` read the project files; `textwrap` keeps labels within boxes; `pandas` handles metadata tables; `NumPy`, `itertools.combinations` and RDKit support the real-ligand geometry panel. Matplotlib supplies the layouts, shapes and connectors, while `IPython.display` embeds the exported images.


In [ ]:
# Project paths and saved metadata.
from pathlib import Path
import json
import textwrap
from itertools import combinations

# Metadata tables and coordinate-based calculations.
import numpy as np
import pandas as pd
from rdkit import Chem

# Shared plotting layouts, drawing primitives and notebook previews.
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle, Polygon
from IPython.display import Image, display

# Load the frozen study metadata before drawing any count-dependent labels.
candidates = [Path.cwd(), Path.cwd() / 'MGT', Path.cwd().parent]
candidates.extend(parent for parent in Path.cwd().parents if parent.name == 'MGT')
ROOT = next((path.resolve() for path in candidates if (path / 'OpenBind_EV-A71_2A').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate the MGT repository.')
DATA_ROOT = ROOT / 'OpenBind_EV-A71_2A' / 'experiment_a_ligand_mgt'
OUTPUT_ROOT = ROOT / 'output' / 'methodology_figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with (DATA_ROOT / 'reports' / 'dataset_metadata.json').open() as handle:
    dataset_metadata = json.load(handle)
with (DATA_ROOT / 'cv_random' / 'cv_metadata.json').open() as handle:
    random_cv_metadata = json.load(handle)
with (DATA_ROOT / 'cv' / 'cv_metadata.json').open() as handle:
    scaffold_cv_metadata = json.load(handle)
# Use the recorded curation counts rather than recounting structures during plotting.
COUNTS = dataset_metadata['counts']
PALETTE = {
    'navy': '#1f3556', 'blue': '#4c78a8', 'light_blue': '#dceaf4',
    'orange': '#f28e2b', 'light_orange': '#fde7cf',
    'green': '#59a14f', 'light_green': '#e2f0df',
    'red': '#e15759', 'light_red': '#f7dddd',
    'purple': '#8f6bb3', 'light_purple': '#ece4f3',
    'teal': '#2a9d8f', 'light_teal': '#d8efec',
    'grey': '#6f7782', 'light_grey': '#eef1f4', 'dark': '#20252b'
}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.titlesize': 17,
                     'figure.titlesize': 20, 'savefig.facecolor': 'white', 'figure.facecolor': 'white'})
# Record the standard figure exports for the manifest at the end.
ARTIFACTS = []

# Use axes-relative coordinates so diagram placement is independent of molecular units.
def canvas(figsize=(16, 9), title=None):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    if title:
        fig.suptitle(title, y=0.985, fontweight='bold', color=PALETTE['dark'])
    return fig, ax

# Keep the title and wrapped description anchored within the same box geometry.
def box(ax, x, y, w, h, title, body='', face='white', edge=None, title_colour=None,
        title_size=12, body_size=10, wrap=28, lw=1.5, radius=0.018, linestyle='solid', zorder=2):
    edge = edge or PALETTE['navy']; title_colour = title_colour or edge
    patch = FancyBboxPatch((x, y), w, h, boxstyle=f'round,pad=0.008,rounding_size={radius}',
                           linewidth=lw, edgecolor=edge, facecolor=face, linestyle=linestyle, zorder=zorder)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h * 0.73, title, ha='center', va='center', fontsize=title_size,
            fontweight='bold', color=title_colour, zorder=zorder + 1)
    if body:
        ax.text(x + w / 2, y + h * 0.37, textwrap.fill(body, wrap), ha='center', va='center',
                fontsize=body_size, color=PALETTE['dark'], linespacing=1.25, zorder=zorder + 1)
    return patch

# Use one connector style across the methodology figures.
def arrow(ax, start, end, colour=None, width=1.8, style='-|>', connection='arc3'):
    patch = FancyArrowPatch(start, end, arrowstyle=style, mutation_scale=15, linewidth=width,
                            color=colour or PALETTE['navy'], connectionstyle=connection, zorder=5)
    ax.add_patch(patch); return patch

# Anchor the panel letter and heading independently of individual boxes.
def panel_label(ax, x, y, label, title):
    ax.text(x, y, label, fontsize=15, fontweight='bold', color='white', ha='center', va='center',
            bbox={'boxstyle': 'round,pad=0.28', 'facecolor': PALETTE['navy'], 'edgecolor': 'none'})
    ax.text(x + 0.035, y, title, fontsize=14, fontweight='bold', color=PALETTE['dark'], ha='left', va='center')

# Export the same figure for print and notebook viewing, then register its filenames.
def save_figure(fig, stem, display_width=1200):
    paths = {}
    for suffix in ('png', 'pdf', 'svg'):
        path = OUTPUT_ROOT / f'{stem}.{suffix}'
        kwargs = {'bbox_inches': 'tight'}
        if suffix == 'png': kwargs['dpi'] = 300
        fig.savefig(path, **kwargs); paths[suffix] = path
        ARTIFACTS.append({'figure': stem, 'format': suffix, 'path': str(path)})
    plt.close(fig); display(Image(filename=str(paths['png']), width=display_width)); return paths

print(f'Figures will be written to: {OUTPUT_ROOT}')

## Figure 2.1 — Overall experimental workflow

**Recommended placement:** Section 2.1. This gives the reader a complete map before the technical detail.

### Code used: Overall workflow layout

I draw the shared preparation stages once, split the diagram into random and scaffold CV branches, and reconnect them at the common training/evaluation stages. Box coordinates and alignment assertions control the layout, not the data partitions.


In [ ]:
# Shared coordinates keep both CV branches comparable without changing their scientific settings.
# -------------------------------------------------------------------------
# Figure 2.1: Top-down branched experimental workflow
# -------------------------------------------------------------------------

fig, ax = canvas(
    (17, 17),
    "Experimental workflow for ligand-only affinity prediction",
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")


# -------------------------------------------------------------------------
# Layout constants
# -------------------------------------------------------------------------

CENTRE_X = 0.50

COMMON_X = 0.30
COMMON_WIDTH = 0.40
COMMON_HEIGHT = 0.060

LEFT_BRANCH_X = 0.045
RIGHT_BRANCH_X = 0.545
BRANCH_WIDTH = 0.410

DESIGN_Y = 0.475
DESIGN_HEIGHT = 0.065

MODEL_PANEL_Y = 0.220
MODEL_PANEL_HEIGHT = 0.225

TRAINING_X = 0.285
TRAINING_Y = 0.095
TRAINING_WIDTH = 0.430
TRAINING_HEIGHT = 0.075

EVALUATION_X = 0.285
EVALUATION_Y = 0.015
EVALUATION_WIDTH = 0.430
EVALUATION_HEIGHT = 0.060

CONNECTOR_COLOUR = PALETTE["navy"]
CONNECTOR_WIDTH = 1.6


# -------------------------------------------------------------------------
# Drawing utilities
# -------------------------------------------------------------------------

def straight_arrow(
    start,
    end,
    colour=CONNECTOR_COLOUR,
    linewidth=CONNECTOR_WIDTH,
    mutation_scale=14,
    zorder=2,
):
    """Draw one completely straight directional arrow."""

    arrow_patch = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=mutation_scale,
        linewidth=linewidth,
        color=colour,
        connectionstyle="arc3,rad=0",
        transform=ax.transAxes,
        clip_on=False,
        zorder=zorder,
    )
    ax.add_patch(arrow_patch)


def straight_line(
    start,
    end,
    colour=CONNECTOR_COLOUR,
    linewidth=CONNECTOR_WIDTH,
    zorder=1,
):
    """Draw one straight connector without an arrowhead."""

    ax.add_line(
        Line2D(
            [start[0], end[0]],
            [start[1], end[1]],
            color=colour,
            linewidth=linewidth,
            solid_capstyle="butt",
            transform=ax.transAxes,
            clip_on=False,
            zorder=zorder,
        )
    )


# Connect box boundaries rather than drawing through the box contents.
def vertical_arrow_between_boxes(
    upper_x,
    upper_y,
    upper_width,
    lower_x,
    lower_y,
    lower_width,
    colour=CONNECTOR_COLOUR,
):
    """Connect the bottom centre of one box to the top centre of another."""

    upper_bottom_centre = (
        upper_x + upper_width / 2,
        upper_y,
    )

    lower_top_centre = (
        lower_x + lower_width / 2,
        lower_y,
    )

    straight_arrow(
        upper_bottom_centre,
        lower_top_centre,
        colour=colour,
    )


# Use compact labels to show all seven configurations inside each CV branch.
def model_chip(
    x,
    y,
    width,
    height,
    label,
    facecolour,
    edgecolour,
):
    """Draw one compact model × five-fold box."""

    patch = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.004,rounding_size=0.007",
        facecolor=facecolour,
        edgecolor=edgecolour,
        linewidth=1.0,
        transform=ax.transAxes,
        clip_on=False,
        zorder=4,
    )
    ax.add_patch(patch)

    ax.text(
        x + width / 2,
        y + height / 2,
        label,
        ha="center",
        va="center",
        fontsize=9.0,
        fontweight="semibold",
        color=PALETTE["navy"],
        transform=ax.transAxes,
        zorder=5,
    )


# Repeat the same model list for each CV design without implying shared fitted weights.
def model_panel(
    x,
    y,
    width,
    height,
    design_name,
    fit_description,
    edgecolour,
):
    """Draw one non-overlapping model panel for a CV design."""

    panel = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.010,rounding_size=0.012",
        facecolor="#fcfcfc",
        edgecolor=edgecolour,
        linewidth=1.4,
        transform=ax.transAxes,
        clip_on=False,
        zorder=1,
    )
    ax.add_patch(panel)

    panel_top = y + height

    # Panel heading.
    ax.text(
        x + width / 2,
        panel_top - 0.026,
        "Seven model configurations × five outer folds",
        ha="center",
        va="center",
        fontsize=12.0,
        fontweight="bold",
        color=edgecolour,
        transform=ax.transAxes,
        zorder=5,
    )

    # Secondary heading.
    ax.text(
        x + width / 2,
        panel_top - 0.052,
        fit_description,
        ha="center",
        va="center",
        fontsize=9.5,
        color="#4d4d4d",
        transform=ax.transAxes,
        zorder=5,
    )

    # Model-chip dimensions.
    horizontal_padding = 0.020
    column_gap = 0.020

    chip_width = (
        width
        - 2 * horizontal_padding
        - column_gap
    ) / 2

    chip_height = 0.028

    left_column_x = x + horizontal_padding

    right_column_x = (
        left_column_x
        + chip_width
        + column_gap
    )

    # Four rows with consistent gaps.
    row_1_y = panel_top - 0.096
    row_2_y = panel_top - 0.132
    row_3_y = panel_top - 0.168
    row_4_y = panel_top - 0.204

    model_specifications = [
        (
            left_column_x,
            row_1_y,
            "Morgan MLP × 5",
            PALETTE["light_grey"],
            "#777777",
        ),
        (
            right_column_x,
            row_1_y,
            "2D GNN × 5",
            PALETTE["light_blue"],
            PALETTE["blue"],
        ),
        (
            left_column_x,
            row_2_y,
            "3D distance GNN × 5",
            PALETTE["light_green"],
            PALETTE["green"],
        ),
        (
            right_column_x,
            row_2_y,
            "3D ALIGNN × 5",
            PALETTE["light_orange"],
            PALETTE["orange"],
        ),
        (
            left_column_x,
            row_3_y,
            "Adapted MGT × 5",
            PALETTE["light_purple"],
            PALETTE["purple"],
        ),
        (
            right_column_x,
            row_3_y,
            "Masked ALIGNN × 5",
            PALETTE["light_red"],
            PALETTE["red"],
        ),
        (
            x + (width - chip_width) / 2,
            row_4_y,
            "Masked MGT × 5",
            "#eee4df",
            "#9a725f",
        ),
    ]

    for (
        chip_x,
        chip_y,
        chip_label,
        chip_face,
        chip_edge,
    ) in model_specifications:
        model_chip(
            chip_x,
            chip_y,
            chip_width,
            chip_height,
            chip_label,
            chip_face,
            chip_edge,
        )


# -------------------------------------------------------------------------
# Shared data-processing pathway
# -------------------------------------------------------------------------

OPENBIND_Y = 0.875
CURATION_Y = 0.790
GEOMETRY_Y = 0.705
GROUPING_Y = 0.620

box(
    ax,
    COMMON_X,
    OPENBIND_Y,
    COMMON_WIDTH,
    COMMON_HEIGHT,
    "1. OpenBind acquisition",
    (
        f"{COUNTS['source_metadata_rows']} crystallographic binding events; "
        "experimentally measured KD values"
    ),
    face=PALETTE["light_blue"],
    edge=PALETTE["blue"],
    wrap=54,
)

vertical_arrow_between_boxes(
    COMMON_X,
    OPENBIND_Y,
    COMMON_WIDTH,
    COMMON_X,
    CURATION_Y + COMMON_HEIGHT,
    COMMON_WIDTH,
)

box(
    ax,
    COMMON_X,
    CURATION_Y,
    COMMON_WIDTH,
    COMMON_HEIGHT,
    "2. Quality-controlled curation",
    (
        "Affinity, covalency, artefact, PoseBusters and canonical "
        "molecular-identity checks"
    ),
    face=PALETTE["light_green"],
    edge=PALETTE["green"],
    wrap=54,
)

vertical_arrow_between_boxes(
    COMMON_X,
    CURATION_Y,
    COMMON_WIDTH,
    COMMON_X,
    GEOMETRY_Y + COMMON_HEIGHT,
    COMMON_WIDTH,
)

box(
    ax,
    COMMON_X,
    GEOMETRY_Y,
    COMMON_WIDTH,
    COMMON_HEIGHT,
    "3. Crystallographic ligand geometry",
    (
        f"{COUNTS['curated_structure_rows']} structures representing "
        f"{COUNTS['benchmark_compound_groups']} compounds; reference SDF "
        "coordinates retained without conformer generation or optimisation"
    ),
    face=PALETTE["light_teal"],
    edge=PALETTE["teal"],
    wrap=58,
)

vertical_arrow_between_boxes(
    COMMON_X,
    GEOMETRY_Y,
    COMMON_WIDTH,
    COMMON_X,
    GROUPING_Y + COMMON_HEIGHT,
    COMMON_WIDTH,
)

box(
    ax,
    COMMON_X,
    GROUPING_Y,
    COMMON_WIDTH,
    COMMON_HEIGHT,
    "4. Leakage-safe compound grouping",
    (
        "Repeated crystallographic structures of the same official "
        "compound remain within one partition"
    ),
    face=PALETTE["light_purple"],
    edge=PALETTE["purple"],
    wrap=54,
)


# -------------------------------------------------------------------------
# Straight branching connector
# -------------------------------------------------------------------------

# Branch only after the common compound-grouping stage.
left_design_centre_x = (
    LEFT_BRANCH_X
    + BRANCH_WIDTH / 2
)

right_design_centre_x = (
    RIGHT_BRANCH_X
    + BRANCH_WIDTH / 2
)

grouping_bottom_y = GROUPING_Y
branch_horizontal_y = 0.568
design_top_y = DESIGN_Y + DESIGN_HEIGHT

# Central vertical line from grouping to branch junction.
straight_line(
    (CENTRE_X, grouping_bottom_y),
    (CENTRE_X, branch_horizontal_y),
)

# Horizontal branch bar.
straight_line(
    (left_design_centre_x, branch_horizontal_y),
    (right_design_centre_x, branch_horizontal_y),
)

# Vertical arrows into each CV-design box.
straight_arrow(
    (left_design_centre_x, branch_horizontal_y),
    (left_design_centre_x, design_top_y),
    colour=PALETTE["blue"],
)

straight_arrow(
    (right_design_centre_x, branch_horizontal_y),
    (right_design_centre_x, design_top_y),
    colour=PALETTE["orange"],
)

# Branch label is positioned above the horizontal connector.
ax.text(
    CENTRE_X,
    branch_horizontal_y + 0.014,
    "Two independent five-fold cross-validation designs",
    ha="center",
    va="bottom",
    fontsize=11.5,
    fontweight="bold",
    color=PALETTE["navy"],
    bbox={
        "boxstyle": "round,pad=0.22",
        "facecolor": "white",
        "edgecolor": "none",
        "alpha": 0.98,
    },
    transform=ax.transAxes,
    zorder=5,
)


# -------------------------------------------------------------------------
# CV-design boxes
# -------------------------------------------------------------------------

box(
    ax,
    LEFT_BRANCH_X,
    DESIGN_Y,
    BRANCH_WIDTH,
    DESIGN_HEIGHT,
    "Random five-fold CV",
    (
        "474 compound groups; pKD-block stratification; compound leakage "
        "prohibited; scaffold overlap allowed and reported"
    ),
    face=PALETTE["light_blue"],
    edge=PALETTE["blue"],
    wrap=49,
)

box(
    ax,
    RIGHT_BRANCH_X,
    DESIGN_Y,
    BRANCH_WIDTH,
    DESIGN_HEIGHT,
    "Scaffold five-fold CV",
    (
        "262 achiral Bemis–Murcko scaffold groups; complete scaffold "
        "families retained together; zero scaffold leakage"
    ),
    face=PALETTE["light_orange"],
    edge=PALETTE["orange"],
    wrap=49,
)


# -------------------------------------------------------------------------
# Straight arrows from CV designs to model panels
# -------------------------------------------------------------------------

left_model_panel_top = (
    MODEL_PANEL_Y
    + MODEL_PANEL_HEIGHT
)

right_model_panel_top = left_model_panel_top

straight_arrow(
    (left_design_centre_x, DESIGN_Y),
    (left_design_centre_x, left_model_panel_top),
    colour=PALETTE["blue"],
)

straight_arrow(
    (right_design_centre_x, DESIGN_Y),
    (right_design_centre_x, right_model_panel_top),
    colour=PALETTE["orange"],
)


# -------------------------------------------------------------------------
# Model panels
# -------------------------------------------------------------------------

model_panel(
    LEFT_BRANCH_X,
    MODEL_PANEL_Y,
    BRANCH_WIDTH,
    MODEL_PANEL_HEIGHT,
    "Random CV",
    "35 independently trained random-CV fits",
    PALETTE["blue"],
)

model_panel(
    RIGHT_BRANCH_X,
    MODEL_PANEL_Y,
    BRANCH_WIDTH,
    MODEL_PANEL_HEIGHT,
    "Scaffold CV",
    "35 independently trained scaffold-CV fits",
    PALETTE["orange"],
)


# -------------------------------------------------------------------------
# Straight convergence connector
# -------------------------------------------------------------------------

# Merge the visual branches at a shared procedure, not a shared model checkpoint.
training_top_y = (
    TRAINING_Y
    + TRAINING_HEIGHT
)

merge_horizontal_y = 0.195

# Straight vertical segments below both model panels.
straight_line(
    (left_design_centre_x, MODEL_PANEL_Y),
    (left_design_centre_x, merge_horizontal_y),
    colour=PALETTE["blue"],
)

straight_line(
    (right_design_centre_x, MODEL_PANEL_Y),
    (right_design_centre_x, merge_horizontal_y),
    colour=PALETTE["orange"],
)

# Shared horizontal merge bar.
straight_line(
    (left_design_centre_x, merge_horizontal_y),
    (right_design_centre_x, merge_horizontal_y),
)

# Single vertical arrow from merge bar to training.
straight_arrow(
    (CENTRE_X, merge_horizontal_y),
    (CENTRE_X, training_top_y),
)


# -------------------------------------------------------------------------
# Matched training
# -------------------------------------------------------------------------

box(
    ax,
    TRAINING_X,
    TRAINING_Y,
    TRAINING_WIDTH,
    TRAINING_HEIGHT,
    "Matched fold-specific training",
    (
        "Applied independently to all 70 fits: training-fold target "
        "normalisation, Huber loss, Adam optimisation and "
        "validation-controlled checkpointing. The branches share a "
        "protocol; their datasets, checkpoints and predictions are not combined."
    ),
    face=PALETTE["light_green"],
    edge=PALETTE["green"],
    wrap=64,
)


# -------------------------------------------------------------------------
# Compound-level evaluation
# -------------------------------------------------------------------------

evaluation_top_y = (
    EVALUATION_Y
    + EVALUATION_HEIGHT
)

straight_arrow(
    (CENTRE_X, TRAINING_Y),
    (CENTRE_X, evaluation_top_y),
)

box(
    ax,
    EVALUATION_X,
    EVALUATION_Y,
    EVALUATION_WIDTH,
    EVALUATION_HEIGHT,
    "Compound-level out-of-fold evaluation",
    (
        "Inverse-transform pKD predictions; average repeated crystal-pose "
        "predictions per compound; report pooled RMSE, R², Pearson and Spearman"
    ),
    face=PALETTE["light_teal"],
    edge=PALETTE["teal"],
    wrap=64,
)


# -------------------------------------------------------------------------
# Layout checks
# -------------------------------------------------------------------------

# Confirm that the two branch panels do not overlap.
assert (
    LEFT_BRANCH_X + BRANCH_WIDTH
    < RIGHT_BRANCH_X
)

# Confirm that every vertical section has a positive gap.
assert OPENBIND_Y > CURATION_Y + COMMON_HEIGHT
assert CURATION_Y > GEOMETRY_Y + COMMON_HEIGHT
assert GEOMETRY_Y > GROUPING_Y + COMMON_HEIGHT
assert branch_horizontal_y > design_top_y
assert DESIGN_Y > left_model_panel_top
assert MODEL_PANEL_Y > merge_horizontal_y
assert merge_horizontal_y > training_top_y
assert TRAINING_Y > evaluation_top_y


# -------------------------------------------------------------------------
# Export
# -------------------------------------------------------------------------

fig.tight_layout(
    rect=(0.015, 0.005, 0.985, 0.965)
)

save_figure(
    fig,
    "figure_2_1_overall_experimental_workflow",
)

plt.show()

**Suggested caption — Figure 2.1.** End-to-end experimental workflow. The OpenBind release was curated into a frozen ligand-only cohort, grouped to prevent repeated-compound leakage, evaluated using random and scaffold five-fold cross-validation, and used in a matched model hierarchy. All comparisons used the same fold-specific training and compound-level out-of-fold evaluation procedure.

## Figure 2.2 — Dataset curation and preservation of crystallographic geometry

**Recommended placement:** Sections 2.2–2.4. This combines selection counts with the geometry-preservation audit.

### Code used: Curation and geometry funnels

I use two staged funnels to separate record filtering from coordinate preservation. Counts come from the loaded metadata where referenced; the stage descriptions and exclusion labels are written explicitly in this cell.


In [ ]:
# The two funnels distinguish excluding records from preserving the retained ligand geometry.
# -------------------------------------------------------------------------
# Figure 2.2: Parallel curation and geometry-preservation funnels
# -------------------------------------------------------------------------

fig, ax = canvas(
    (17, 11),
    "Dataset curation and crystallographic-geometry preservation",
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")


# -------------------------------------------------------------------------
# Funnel data
# -------------------------------------------------------------------------

# These stage specifications supply the funnel labels and order.
curation_stages = [
    {
        "title": "1. Structure-file audit",
        "description": (
            f"{COUNTS['source_metadata_rows']} metadata rows mapped to "
            "3,700 expected structure files; no missing or unexpected files"
        ),
    },
    {
        "title": "2. Affinity eligibility",
        "description": (
            "Experimental pKD required and the complex must be represented "
            "in the official OpenBind affinity reference"
        ),
    },
    {
        "title": "3. Structural quality filters",
        "description": (
            "Retain non-covalent ligands without suspected artefacts and "
            "with a valid reference PoseBusters result"
        ),
    },
    {
        "title": "4. Molecular-identity audit",
        "description": (
            "RDKit isomeric canonical identity from the reference SDF "
            "must agree exactly with the metadata identity"
        ),
    },
    {
        "title": "5. Exclusion audit",
        "description": (
            "Remove 304 structures while retaining every excluded record "
            "and its overlapping exclusion reasons"
        ),
    },
]

# Describe the coordinate-preserving conversion separately from quality filtering.
geometry_stages = [
    {
        "title": "1. Crystallographic source",
        "description": (
            "Load ligand_ref.sdf with its experimentally observed bound "
            "coordinates, atom identities and chemical bonds"
        ),
    },
    {
        "title": "2. Molecule validation",
        "description": (
            "Require one valid RDKit molecule and one crystallographic "
            "conformer for every retained structure"
        ),
    },
    {
        "title": "3. Model-specific input routing",
        "description": (
            "The 2D GNN, 3D distance GNN and ALIGNN read the reference "
            "SDF directly, retaining bond orders and coordinates"
        ),
    },
    {
        "title": "4. Adapted MGT compatibility",
        "description": (
            "Export a ligand-only PDB while preserving atom identities, "
            "coordinates and any explicit hydrogens present"
        ),
    },
    {
        "title": "5. Round-trip geometry audit",
        "description": (
            "Require an unchanged atom count and maximum coordinate "
            "deviation no greater than 0.00011 Å"
        ),
    },
]


# -------------------------------------------------------------------------
# Funnel appearance
# -------------------------------------------------------------------------

left_colours = [
    "#d9e8f5",
    "#c3daed",
    "#a8c9e3",
    "#83afd3",
    "#5f92bf",
]

right_colours = [
    "#dcefd9",
    "#c6e6c0",
    "#a5d89d",
    "#78c06f",
    "#4e9d49",
]

left_edge = PALETTE["blue"]
right_edge = PALETTE["green"]

body_face = "#ffffff"
body_text_colour = "#20252d"
outline_width = 1.15


# -------------------------------------------------------------------------
# Drawing utilities
# -------------------------------------------------------------------------

def vertical_arrow(
    x,
    y_start,
    y_end,
    colour=PALETTE["navy"],
):
    """Draw one straight vertical arrow."""

    patch = FancyArrowPatch(
        (x, y_start),
        (x, y_end),
        arrowstyle="-|>",
        mutation_scale=14,
        linewidth=1.5,
        color=colour,
        connectionstyle="arc3,rad=0",
        transform=ax.transAxes,
        clip_on=False,
        zorder=4,
    )
    ax.add_patch(patch)


# Interpolate the funnel width by stage while keeping each text label inside its segment.
def draw_funnel(
    centre_x,
    funnel_top,
    funnel_bottom,
    maximum_width,
    minimum_width,
    stages,
    stage_colours,
    edge_colour,
    input_label,
    output_label,
):
    """
    Draw one vertically tapering methodology funnel.

    Each stage contains a coloured heading band followed by a white
    explanatory band. All widths and text positions are calculated from
    the same funnel geometry to prevent overlaps.
    """

    number_of_stages = len(stages)
    total_height = funnel_top - funnel_bottom
    stage_height = total_height / number_of_stages

    # Fraction of each stage assigned to the coloured heading.
    heading_fraction = 0.32

    def width_at(stage_position):
        """Return funnel width at a fractional position from top to bottom."""

        return maximum_width - (
            maximum_width - minimum_width
        ) * stage_position

    for stage_index, stage in enumerate(stages):

        top_position = stage_index / number_of_stages
        bottom_position = (
            stage_index + 1
        ) / number_of_stages

        heading_bottom_position = (
            stage_index + heading_fraction
        ) / number_of_stages

        width_top = width_at(top_position)
        width_heading_bottom = width_at(
            heading_bottom_position
        )
        width_bottom = width_at(bottom_position)

        y_top = (
            funnel_top
            - stage_index * stage_height
        )

        y_bottom = y_top - stage_height

        y_heading_bottom = (
            y_top
            - stage_height * heading_fraction
        )

        # Horizontal coordinates of the heading band.
        top_left = centre_x - width_top / 2
        top_right = centre_x + width_top / 2

        heading_left = (
            centre_x
            - width_heading_bottom / 2
        )
        heading_right = (
            centre_x
            + width_heading_bottom / 2
        )

        # Horizontal coordinates of the description band.
        bottom_left = (
            centre_x
            - width_bottom / 2
        )
        bottom_right = (
            centre_x
            + width_bottom / 2
        )

        # Coloured heading band.
        heading_polygon = Polygon(
            [
                (top_left, y_top),
                (top_right, y_top),
                (
                    heading_right,
                    y_heading_bottom,
                ),
                (
                    heading_left,
                    y_heading_bottom,
                ),
            ],
            closed=True,
            facecolor=stage_colours[stage_index],
            edgecolor=edge_colour,
            linewidth=outline_width,
            transform=ax.transAxes,
            clip_on=False,
            zorder=2,
        )
        ax.add_patch(heading_polygon)

        # White description band.
        description_polygon = Polygon(
            [
                (
                    heading_left,
                    y_heading_bottom,
                ),
                (
                    heading_right,
                    y_heading_bottom,
                ),
                (
                    bottom_right,
                    y_bottom,
                ),
                (
                    bottom_left,
                    y_bottom,
                ),
            ],
            closed=True,
            facecolor=body_face,
            edgecolor=edge_colour,
            linewidth=outline_width,
            transform=ax.transAxes,
            clip_on=False,
            zorder=2,
        )
        ax.add_patch(description_polygon)

        # White title text is used only on the darkest final band.
        title_colour = (
            "white"
            if stage_index == number_of_stages - 1
            else PALETTE["navy"]
        )

        ax.text(
            centre_x,
            (
                y_top
                + y_heading_bottom
            ) / 2,
            stage["title"],
            ha="center",
            va="center",
            fontsize=10.5,
            fontweight="bold",
            color=title_colour,
            transform=ax.transAxes,
            zorder=3,
        )

        # Narrower lower stages receive a smaller wrapping width.
        wrapping_width = max(
            27,
            int(103 * width_bottom),
        )

        wrapped_description = textwrap.fill(
            stage["description"],
            width=wrapping_width,
        )

        ax.text(
            centre_x,
            (
                y_heading_bottom
                + y_bottom
            ) / 2,
            wrapped_description,
            ha="center",
            va="center",
            fontsize=9.1,
            color=body_text_colour,
            linespacing=1.12,
            transform=ax.transAxes,
            zorder=3,
        )

    # Straight input arrow.
    vertical_arrow(
        centre_x,
        funnel_top + 0.065,
        funnel_top + 0.008,
        colour=edge_colour,
    )

    # Input label.
    ax.text(
        centre_x,
        funnel_top + 0.086,
        input_label,
        ha="center",
        va="center",
        fontsize=12.5,
        fontweight="bold",
        color=edge_colour,
        transform=ax.transAxes,
    )

    # Straight output arrow.
    vertical_arrow(
        centre_x,
        funnel_bottom - 0.005,
        funnel_bottom - 0.050,
        colour=edge_colour,
    )

    # Output label.
    ax.text(
        centre_x,
        funnel_bottom - 0.074,
        output_label,
        ha="center",
        va="center",
        fontsize=12.5,
        fontweight="bold",
        color=edge_colour,
        transform=ax.transAxes,
    )


# -------------------------------------------------------------------------
# Panel headings
# -------------------------------------------------------------------------

panel_label(
    ax,
    0.055,
    0.925,
    "A",
    "Curation and audit trail",
)

panel_label(
    ax,
    0.555,
    0.925,
    "B",
    "Crystallographic geometry retained for ligand-only models",
)


# -------------------------------------------------------------------------
# Draw the parallel funnels
# -------------------------------------------------------------------------

FUNNEL_TOP = 0.795
FUNNEL_BOTTOM = 0.255

draw_funnel(
    centre_x=0.255,
    funnel_top=FUNNEL_TOP,
    funnel_bottom=FUNNEL_BOTTOM,
    maximum_width=0.405,
    minimum_width=0.230,
    stages=curation_stages,
    stage_colours=left_colours,
    edge_colour=left_edge,
    input_label="OpenBind structure–affinity release",
    output_label=(
        f"Frozen cohort: "
        f"{COUNTS['curated_structure_rows']} structures / "
        f"{COUNTS['benchmark_compound_groups']} compounds"
    ),
)

draw_funnel(
    centre_x=0.745,
    funnel_top=FUNNEL_TOP,
    funnel_bottom=FUNNEL_BOTTOM,
    maximum_width=0.405,
    minimum_width=0.230,
    stages=geometry_stages,
    stage_colours=right_colours,
    edge_colour=right_edge,
    input_label="Retained ligand_ref.sdf structures",
    output_label="Geometry-preserved model inputs",
)


# -------------------------------------------------------------------------
# Supporting audit notes
# -------------------------------------------------------------------------

ax.text(
    0.255,
    0.105,
    (
        "Exclusion reasons overlap:\n"
        "missing pKD 276 • absent from official reference 276 • "
        "suspected artefact 39\n"
        "failed reference PoseBusters 28 • covalent ligand 2"
    ),
    ha="center",
    va="center",
    fontsize=9.5,
    color=PALETTE["red"],
    linespacing=1.25,
    bbox={
        "boxstyle": "round,pad=0.38",
        "facecolor": "#fff7f7",
        "edgecolor": PALETTE["red"],
        "linewidth": 0.9,
        "alpha": 0.96,
    },
    transform=ax.transAxes,
)

ax.text(
    0.745,
    0.105,
    (
        "No conformer generation • no energy minimisation\n"
        "No coordinate relaxation • protein coordinates omitted"
    ),
    ha="center",
    va="center",
    fontsize=10.0,
    fontweight="semibold",
    color=PALETTE["red"],
    linespacing=1.25,
    bbox={
        "boxstyle": "round,pad=0.38",
        "facecolor": "#fff7f7",
        "edgecolor": PALETTE["red"],
        "linewidth": 0.9,
        "alpha": 0.96,
    },
    transform=ax.transAxes,
)


# -------------------------------------------------------------------------
# Centre divider
# -------------------------------------------------------------------------

ax.plot(
    [0.50, 0.50],
    [0.085, 0.875],
    color="#d8d8d8",
    linewidth=0.8,
    linestyle="--",
    transform=ax.transAxes,
    zorder=0,
)


# -------------------------------------------------------------------------
# Layout checks
# -------------------------------------------------------------------------

# Verify that the two funnels remain separated.
assert (
    0.255 + 0.405 / 2
    < 0.745 - 0.405 / 2
)

# Verify sufficient vertical space for labels and audit notes.
assert FUNNEL_TOP + 0.086 < 0.90
assert FUNNEL_BOTTOM - 0.074 > 0.15


# -------------------------------------------------------------------------
# Export
# -------------------------------------------------------------------------

fig.tight_layout(
    rect=(0.015, 0.01, 0.985, 0.955)
)

save_figure(
    fig,
    "figure_2_2_curation_and_geometry",
)

plt.show()

**Suggested caption — Figure 2.2.** Dataset curation and geometry-preservation procedure. All metadata records were mapped to their expected files before predefined affinity, covalency, artefact, pose-validity and identity filters were applied. Experimental coordinates were read directly for the controlled models and round-trip checked after ligand-only PDB conversion for adapted MGT. Exclusion reason counts are non-exclusive.

## Figure 2.3 — Five-fold cross-validation and leakage controls

**Recommended placement:** Sections 2.5–2.7. This is essential because validation at each epoch is not the same as cross-validation.

### Code used: Cross-validation schematic

I draw a five-fold rotation matrix and an approximate training/validation/test allocation bar. This illustrates the saved CV design; it does not generate folds or calculate their actual sizes.


In [ ]:
# This matrix shows outer-fold roles; actual memberships remain in the saved CSV manifests.
# -------------------------------------------------------------------------
# Figure 2.3: Leakage-controlled five-fold cross-validation
# -------------------------------------------------------------------------

fig, ax = canvas(
    (17, 10),
    "Random and scaffold five-fold cross-validation",
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")


# -------------------------------------------------------------------------
# Panel headings
# -------------------------------------------------------------------------

panel_label(
    ax,
    0.035,
    0.920,
    "A",
    "Outer-fold rotation",
)

panel_label(
    ax,
    0.565,
    0.920,
    "B",
    "Grouping and leakage-control rules",
)


# =========================================================================
# Panel A: Outer-fold rotation
# =========================================================================

matrix_x = 0.095
matrix_y = 0.555

cell_width = 0.077
cell_height = 0.052
cell_gap_x = 0.005
cell_gap_y = 0.006

matrix_centre_x = (
    matrix_x
    + 5 * cell_width / 2
)

# -------------------------------------------------------------------------
# Column headings
# -------------------------------------------------------------------------

for column in range(5):

    column_centre = (
        matrix_x
        + column * cell_width
        + (cell_width - cell_gap_x) / 2
    )

    ax.text(
        column_centre,
        matrix_y + 5 * cell_height + 0.028,
        f"Group {column}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color=PALETTE["dark"],
    )


# -------------------------------------------------------------------------
# Rotation matrix
# -------------------------------------------------------------------------

# Mark one test fold per row; the other folds provide development data.
for outer_run in range(5):

    row_y = (
        matrix_y
        + (4 - outer_run) * cell_height
    )

    # Row label.
    ax.text(
        matrix_x - 0.016,
        row_y + (cell_height - cell_gap_y) / 2,
        f"Outer run {outer_run}",
        ha="right",
        va="center",
        fontsize=9.8,
        color=PALETTE["dark"],
    )

    for column in range(5):

        is_test = outer_run == column

        cell_x = (
            matrix_x
            + column * cell_width
        )

        cell = Rectangle(
            (
                cell_x,
                row_y,
            ),
            cell_width - cell_gap_x,
            cell_height - cell_gap_y,
            facecolor=(
                PALETTE["light_red"]
                if is_test
                else PALETTE["light_grey"]
            ),
            edgecolor=(
                PALETTE["red"]
                if is_test
                else "#aab2bb"
            ),
            linewidth=1.1,
        )
        ax.add_patch(cell)

        ax.text(
            cell_x + (cell_width - cell_gap_x) / 2,
            row_y + (cell_height - cell_gap_y) / 2,
            (
                "TEST"
                if is_test
                else "DEV"
            ),
            ha="center",
            va="center",
            fontsize=8.5,
            fontweight=(
                "bold"
                if is_test
                else "normal"
            ),
            color=(
                PALETTE["red"]
                if is_test
                else PALETTE["grey"]
            ),
        )


# -------------------------------------------------------------------------
# Explanation below the rotation matrix
# -------------------------------------------------------------------------

ax.text(
    matrix_centre_x,
    0.490,
    "Each compound is external test data exactly once",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
    color=PALETTE["navy"],
)

arrow(
    ax,
    (
        matrix_centre_x,
        0.470,
    ),
    (
        matrix_centre_x,
        0.405,
    ),
)


# -------------------------------------------------------------------------
# Train/validation/test allocation bar
# -------------------------------------------------------------------------

bar_x = 0.065
bar_y = 0.295
bar_width = 0.445
bar_height = 0.063

# The 64/16/20 proportions are a schematic summary, not measured fold counts.
split_segments = [
    {
        "name": "Training",
        "percentage": "≈64%",
        "fraction": 0.64,
        "colour": PALETTE["blue"],
    },
    {
        "name": "Validation",
        "percentage": "≈16%",
        "fraction": 0.16,
        "colour": PALETTE["green"],
    },
    {
        "name": "Test",
        "percentage": "≈20%",
        "fraction": 0.20,
        "colour": PALETTE["red"],
    },
]

cursor = bar_x

for segment in split_segments:

    segment_width = (
        bar_width
        * segment["fraction"]
    )

    segment_patch = Rectangle(
        (
            cursor,
            bar_y,
        ),
        segment_width,
        bar_height,
        facecolor=segment["colour"],
        edgecolor="white",
        linewidth=1.5,
        zorder=2,
    )
    ax.add_patch(segment_patch)

    segment_centre = (
        cursor
        + segment_width / 2
    )

    # Put the category name above the bar, where space is unrestricted.
    ax.text(
        segment_centre,
        bar_y + bar_height + 0.017,
        segment["name"],
        ha="center",
        va="bottom",
        fontsize=9.8,
        fontweight="bold",
        color=segment["colour"],
        zorder=3,
    )

    # Only the short percentage remains inside the segment.
    percentage_text = ax.text(
        segment_centre,
        bar_y + bar_height / 2,
        segment["percentage"],
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color="white",
        zorder=3,
    )

    # Prevent the percentage from extending beyond its coloured segment.
    percentage_text.set_clip_path(
        segment_patch
    )

    cursor += segment_width


# Explanation below the allocation bar.
ax.text(
    matrix_centre_x,
    0.247,
    (
        "Validation selects the checkpoint;\n"
        "the outer test fold remains untouched"
    ),
    ha="center",
    va="top",
    fontsize=10.2,
    color=PALETTE["dark"],
    linespacing=1.25,
)


# =========================================================================
# Panel B: Grouping rules
# =========================================================================

# -------------------------------------------------------------------------
# Random and scaffold grouping cards
# -------------------------------------------------------------------------

box(
    ax,
    0.560,
    0.630,
    0.195,
    0.190,
    "Random CV",
    (
        "474 compound groups; pKD-block stratification; seeded allocation; "
        "scaffold overlap allowed and reported"
    ),
    face=PALETTE["light_blue"],
    edge=PALETTE["blue"],
    wrap=27,
    title_size=13,
    body_size=9.6,
)

box(
    ax,
    0.775,
    0.630,
    0.195,
    0.190,
    "Scaffold CV",
    (
        "262 achiral Bemis–Murcko groups; whole scaffold families "
        "indivisible; zero scaffold overlap"
    ),
    face=PALETTE["light_orange"],
    edge=PALETTE["orange"],
    wrap=27,
    title_size=13,
    body_size=9.6,
)


# -------------------------------------------------------------------------
# Shared grouping rule
# -------------------------------------------------------------------------

box(
    ax,
    0.560,
    0.500,
    0.410,
    0.085,
    "Shared compound-group rule",
    (
        "Repeated crystallographic structures remain with their official "
        "compound group in every partition"
    ),
    face=PALETTE["light_purple"],
    edge=PALETTE["purple"],
    wrap=56,
    title_size=12,
    body_size=9.8,
)


# -------------------------------------------------------------------------
# Leakage boundary
# -------------------------------------------------------------------------

box(
    ax,
    0.560,
    0.255,
    0.410,
    0.190,
    "Leakage boundary",
    (
        "Training only: target mean and standard deviation, atom-feature "
        "masking and gradient optimisation. Validation only: learning-rate "
        "scheduling, early stopping and checkpoint selection. Outer test "
        "only: final prediction after model selection."
    ),
    face="#fff7df",
    edge="#c79a1b",
    wrap=58,
    body_size=10.0,
    title_size=13,
)


# =========================================================================
# Reproducibility banner
# =========================================================================

ax.text(
    0.50,
    0.095,
    (
        "Frozen manifests  •  SHA-256 checksums  •  seed 123  •  "
        "zero compound leakage"
    ),
    ha="center",
    va="center",
    fontsize=12.5,
    fontweight="bold",
    color="white",
    bbox={
        "boxstyle": "round,pad=0.50",
        "facecolor": PALETTE["navy"],
        "edgecolor": "none",
    },
)


# -------------------------------------------------------------------------
# Layout checks
# -------------------------------------------------------------------------

# Verify that the train/validation/test segments sum exactly to the bar width.
assert abs(
    sum(
        segment["fraction"]
        for segment in split_segments
    )
    - 1.0
) < 1e-12

# Verify that the grouping cards do not overlap.
assert (
    0.560 + 0.195
    < 0.775
)

# Verify that the right-side cards remain within the canvas.
assert (
    0.775 + 0.195
    <= 1.0
)

# Verify positive vertical gaps between the right-side sections.
assert 0.630 > 0.500 + 0.085
assert 0.500 > 0.255 + 0.190


# -------------------------------------------------------------------------
# Export
# -------------------------------------------------------------------------

fig.tight_layout(
    rect=(
        0.01,
        0.01,
        0.99,
        0.95,
    )
)

save_figure(
    fig,
    "figure_2_3_five_fold_cross_validation",
)

plt.show()

**Suggested caption — Figure 2.3.** Five-fold cross-validation and leakage controls. Each compound was assigned to one outer test fold exactly once. Remaining development compounds were repartitioned into training and validation data. Random CV preserved compound groups while permitting scaffold overlap; scaffold CV kept complete achiral Bemis–Murcko families together. Target normalisation, masking and optimisation used training data only.

## Figure 2.4 — Controlled model hierarchy and exact ablation logic

**Recommended placement:** Section 2.8. This shows which information is added at each comparison.

### Code used: Model-comparison diagram

I store each model's input description and comparison role in a row specification, then draw a consistently sized table. The layer descriptions and parameter labels are entered here rather than read from checkpoints.


In [ ]:
# Model-row specifications supply the displayed content; column widths only affect readability.
# -------------------------------------------------------------------------
# Figure 2.4: Tabular hierarchy of molecular representations
# -------------------------------------------------------------------------

fig, ax = canvas(
    (18, 8.5),
    "Controlled hierarchy of ligand representations",
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")


# -------------------------------------------------------------------------
# Model information
# -------------------------------------------------------------------------

# Keep model descriptions and their comparison roles together in each row.
model_rows = [
    {
        "step": "1",
        "model": "Morgan MLP",
        "construction": (
            "Canonical SMILES → radius-2, 2,048-bit Morgan fingerprint → "
            "MLP 2,048 → 128 → 64 → 1"
        ),
        "comparison": (
            "Fixed two-dimensional molecular substructure representation"
        ),
        "parameters": "0.27M",
        "masked": "—",
        "edge": PALETTE["grey"],
        "face": PALETTE["light_grey"],
    },
    {
        "step": "2",
        "model": "2D GNN",
        "construction": (
            "152-dimensional atom features and 12-dimensional bond features; "
            "chemical-bond graph → three edge-gated layers → mean pooling"
        ),
        "comparison": (
            "Introduces learned atom–bond topology relative to the fixed fingerprint"
        ),
        "parameters": "4.09M",
        "masked": "—",
        "edge": PALETTE["blue"],
        "face": PALETTE["light_blue"],
    },
    {
        "step": "3",
        "model": "3D distance GNN",
        "construction": (
            "2D chemical graph plus crystallographic spatial neighbours within "
            "5 Å; maximum 32 neighbours; 40-bin distance RBF expansion"
        ),
        "comparison": (
            "Adds crystallographic pair distances while retaining the same "
            "atom–bond representation and graph-update layers"
        ),
        "parameters": "4.10M",
        "masked": "—",
        "edge": PALETTE["green"],
        "face": PALETTE["light_green"],
    },
    {
        "step": "4",
        "model": "3D ALIGNN",
        "construction": (
            "Distance graph plus DGL line graph; bond-angle cosine represented "
            "by a 40-bin RBF; three ALIGNN angle–edge–atom updates"
        ),
        "comparison": (
            "Adds explicit angular geometry to the crystallographic "
            "3D distance representation"
        ),
        "parameters": "8.12M",
        "masked": (
            "Masked ALIGNN\n20% reconstruction"
        ),
        "edge": PALETTE["orange"],
        "face": PALETTE["light_orange"],
    },
    {
        "step": "5",
        "model": "Adapted MGT",
        "construction": (
            "90-dimensional elemental state plus 10-dimensional Laplacian "
            "encoding; local distance graph, angular line graph, full Coulomb "
            "graph, multi-head attention and one complete MGT encoder block"
        ),
        "comparison": (
            "Introduces wider Coulomb attention and the complete MGT encoder; "
            "this is a broader architectural comparison rather than a "
            "single-component ablation"
        ),
        "parameters": "13.70M",
        "masked": (
            "Masked MGT\n20% reconstruction"
        ),
        "edge": PALETTE["purple"],
        "face": PALETTE["light_purple"],
    },
]


# -------------------------------------------------------------------------
# Table geometry
# -------------------------------------------------------------------------

table_x = 0.025
table_top = 0.880
table_width = 0.950

header_height = 0.070
row_height = 0.139

column_definitions = [
    {
        "name": "Step",
        "width": 0.055,
        "wrap": 8,
    },
    {
        "name": "Model",
        "width": 0.140,
        "wrap": 18,
    },
    {
        "name": "Input representation and model construction",
        "width": 0.370,
        "wrap": 58,
    },
    {
        "name": "Scientific comparison",
        "width": 0.220,
        "wrap": 34,
    },
    {
        "name": "Parameters",
        "width": 0.075,
        "wrap": 12,
    },
    {
        "name": "Masked-pretraining variant",
        "width": 0.090,
        "wrap": 17,
    },
]

# Confirm that the columns occupy the requested table width.
assert abs(
    sum(
        column["width"]
        for column in column_definitions
    )
    - table_width
) < 1e-12


# -------------------------------------------------------------------------
# Column coordinates
# -------------------------------------------------------------------------

# Derive column positions cumulatively to avoid drifting table boundaries.
column_left_edges = []

current_x = table_x

for column in column_definitions:
    column_left_edges.append(current_x)
    current_x += column["width"]


# -------------------------------------------------------------------------
# Header
# -------------------------------------------------------------------------

header_bottom = (
    table_top
    - header_height
)

for column_index, column in enumerate(column_definitions):

    column_x = column_left_edges[column_index]

    header_cell = Rectangle(
        (
            column_x,
            header_bottom,
        ),
        column["width"],
        header_height,
        facecolor=PALETTE["navy"],
        edgecolor="white",
        linewidth=1.2,
        zorder=2,
    )
    ax.add_patch(header_cell)

    header_text = textwrap.fill(
        column["name"],
        width=column["wrap"],
    )

    ax.text(
        column_x + column["width"] / 2,
        header_bottom + header_height / 2,
        header_text,
        ha="center",
        va="center",
        fontsize=10.2,
        fontweight="bold",
        color="white",
        linespacing=1.10,
        zorder=3,
    )


# -------------------------------------------------------------------------
# Body rows
# -------------------------------------------------------------------------

for row_index, model in enumerate(model_rows):

    row_top = (
        header_bottom
        - row_index * row_height
    )

    row_bottom = (
        row_top
        - row_height
    )

    default_background = (
        "#ffffff"
        if row_index % 2 == 0
        else "#f8f9fb"
    )

    row_values = [
        model["step"],
        model["model"],
        model["construction"],
        model["comparison"],
        model["parameters"],
        model["masked"],
    ]

    for column_index, (
        column,
        value,
    ) in enumerate(
        zip(
            column_definitions,
            row_values,
        )
    ):

        column_x = column_left_edges[column_index]

        # The model-name cell receives the model-specific colour.
        if column_index == 1:
            cell_face = model["face"]
            cell_edge = model["edge"]
            cell_linewidth = 1.3
        else:
            cell_face = default_background
            cell_edge = "#d9dee3"
            cell_linewidth = 0.9

        body_cell = Rectangle(
            (
                column_x,
                row_bottom,
            ),
            column["width"],
            row_height,
            facecolor=cell_face,
            edgecolor=cell_edge,
            linewidth=cell_linewidth,
            zorder=1,
        )
        ax.add_patch(body_cell)

        # Step numbers are shown as coloured circular labels.
        if column_index == 0:

            ax.text(
                column_x + column["width"] / 2,
                row_bottom + row_height / 2,
                value,
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color="white",
                bbox={
                    "boxstyle": "circle,pad=0.34",
                    "facecolor": model["edge"],
                    "edgecolor": "white",
                    "linewidth": 1.0,
                },
                zorder=3,
            )

            continue

        wrapped_value = textwrap.fill(
            value,
            width=column["wrap"],
        )

        # Formatting differs slightly by column.
        if column_index == 1:
            font_size = 11.0
            font_weight = "bold"
            font_colour = model["edge"]

        elif column_index == 2:
            font_size = 9.2
            font_weight = "normal"
            font_colour = PALETTE["dark"]

        elif column_index == 3:
            font_size = 9.1
            font_weight = "normal"
            font_colour = PALETTE["dark"]

        elif column_index == 4:
            font_size = 10.3
            font_weight = "bold"
            font_colour = model["edge"]

        else:
            font_size = 8.8
            font_weight = (
                "semibold"
                if value != "—"
                else "normal"
            )
            font_colour = (
                model["edge"]
                if value != "—"
                else "#b8bdc2"
            )

        text_artist = ax.text(
            column_x + column["width"] / 2,
            row_bottom + row_height / 2,
            wrapped_value,
            ha="center",
            va="center",
            fontsize=font_size,
            fontweight=font_weight,
            color=font_colour,
            linespacing=1.15,
            zorder=3,
        )

        # Keep every text object inside its cell.
        text_artist.set_clip_path(
            body_cell
        )


# -------------------------------------------------------------------------
# Hierarchy markers between rows
# -------------------------------------------------------------------------

step_column_centre = (
    column_left_edges[0]
    + column_definitions[0]["width"] / 2
)

for row_index in range(
    len(model_rows) - 1
):

    boundary_y = (
        header_bottom
        - (row_index + 1) * row_height
    )

    ax.annotate(
        "",
        xy=(
            step_column_centre,
            boundary_y - 0.012,
        ),
        xytext=(
            step_column_centre,
            boundary_y + 0.012,
        ),
        arrowprops={
            "arrowstyle": "-|>",
            "color": PALETTE["navy"],
            "linewidth": 1.0,
            "mutation_scale": 10,
        },
        zorder=4,
    )


# -------------------------------------------------------------------------
# Scientific interpretation
# -------------------------------------------------------------------------

ax.text(
    0.50,
    0.057,
    (
        "Controlled ablations: 2D GNN → 3D distance GNN isolates "
        "crystallographic distances; 3D distance GNN → 3D ALIGNN isolates "
        "angular information.\n"
        "The adapted MGT is a broader architecture comparison because it "
        "also changes the atom representation, local-graph settings, "
        "positional encoding and encoder structure."
    ),
    ha="center",
    va="center",
    fontsize=10.5,
    color=PALETTE["navy"],
    fontweight="semibold",
    linespacing=1.30,
    bbox={
        "boxstyle": "round,pad=0.42",
        "facecolor": "#f6f8fb",
        "edgecolor": "#c7d0da",
        "linewidth": 0.9,
    },
)


# -------------------------------------------------------------------------
# Layout checks
# -------------------------------------------------------------------------

table_bottom = (
    header_bottom
    - len(model_rows) * row_height
)

# Ensure the table does not overlap the interpretation note.
assert table_bottom > 0.105

# Ensure every column remains inside the canvas.
assert current_x <= 1.0

# Ensure all model rows have the required fields.
assert all(
    set(model.keys())
    >= {
        "step",
        "model",
        "construction",
        "comparison",
        "parameters",
        "masked",
        "edge",
        "face",
    }
    for model in model_rows
)


# -------------------------------------------------------------------------
# Export
# -------------------------------------------------------------------------

fig.tight_layout(
    rect=(
        0.01,
        0.01,
        0.99,
        0.95,
    )
)

save_figure(
    fig,
    "figure_2_4_controlled_model_hierarchy",
)

plt.show()

**Suggested caption — Figure 2.4.** Controlled molecular-representation hierarchy. Models progressively introduced learned atom–bond connectivity, crystallographic distances, angular line-graph information and wider-graph Coulomb attention. Masked variants retained the corresponding supervised architecture after training-fold-only atom-feature reconstruction. Parameter counts exclude the discarded reconstruction decoder.

## Figure 2.5 — Detailed adapted-MGT architecture

**Recommended placement:** Sections 2.9–2.11. This central model-design figure distinguishes local, angular and wider graph views.

### Code used: Adapted MGT architecture

I place the input graphs, encoder stages and ligand-level readout in aligned columns. The routing helpers keep connections outside the boxes; no tensors pass through a model in this figure cell.


In [ ]:
# Keep the three graph inputs separate until the shared hidden-space embedding stage.
# -------------------------------------------------------------------------
# Figure 2.5: Adapted ligand Molecular Graph Transformer
# -------------------------------------------------------------------------

fig, ax = canvas(
    (19, 10.5),
    "Adapted ligand Molecular Graph Transformer",
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")


# -------------------------------------------------------------------------
# Layout constants
# -------------------------------------------------------------------------

# Panel A: input representations.
INPUT_X = 0.030
INPUT_WIDTH = 0.270
INPUT_HEIGHT = 0.110
INPUT_BUS_X = 0.340

# Panel B: MGT encoder.
ENCODER_X = 0.390
ENCODER_WIDTH = 0.340

ENCODER_INPUT_Y = 0.750
ENCODER_INPUT_HEIGHT = 0.065

ENCODER_STEP_HEIGHT = 0.090

# Panel C: regression pathway.
OUTPUT_X = 0.800
OUTPUT_WIDTH = 0.170
OUTPUT_BOX_HEIGHT = 0.085

# Shared connector settings.
CONNECTOR_COLOUR = PALETTE["navy"]
CONNECTOR_WIDTH = 1.5


# -------------------------------------------------------------------------
# Straight connector utilities
# -------------------------------------------------------------------------

def straight_line(
    start,
    end,
    colour=CONNECTOR_COLOUR,
    linewidth=CONNECTOR_WIDTH,
    zorder=2,
):
    """Draw one completely straight line without an arrowhead."""

    line = Line2D(
        [start[0], end[0]],
        [start[1], end[1]],
        color=colour,
        linewidth=linewidth,
        solid_capstyle="butt",
        transform=ax.transAxes,
        clip_on=False,
        zorder=zorder,
    )
    ax.add_line(line)


def straight_arrow(
    start,
    end,
    colour=CONNECTOR_COLOUR,
    linewidth=CONNECTOR_WIDTH,
    mutation_scale=14,
    zorder=3,
):
    """Draw one completely straight directional arrow."""

    patch = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=mutation_scale,
        linewidth=linewidth,
        color=colour,
        connectionstyle="arc3,rad=0",
        transform=ax.transAxes,
        clip_on=False,
        zorder=zorder,
    )
    ax.add_patch(patch)


# -------------------------------------------------------------------------
# Panel headings
# -------------------------------------------------------------------------

panel_label(
    ax,
    0.030,
    0.920,
    "A",
    "Input features and three graph views",
)

panel_label(
    ax,
    0.390,
    0.920,
    "B",
    "One complete MGT encoder block",
)

panel_label(
    ax,
    0.800,
    0.920,
    "C",
    "Ligand-level regression",
)

# Place the hidden-width description between the panel heading and the
# embedded-input box.
ax.text(
    ENCODER_X + ENCODER_WIDTH / 2,
    0.855,
    "Shared encoder hidden width = 512",
    ha="center",
    va="center",
    fontsize=11.5,
    fontweight="bold",
    color=PALETTE["navy"],
    transform=ax.transAxes,
)


# =========================================================================
# Panel A: atom features and graph representations
# =========================================================================

# Each input specification controls one feature-view box and its colour.
input_specs = [
    {
        "y": 0.700,
        "title": "Atom state",
        "body": (
            "90-dimensional elemental vector plus deterministic "
            "10-dimensional Laplacian positional encoding"
        ),
        "edge": PALETTE["blue"],
        "face": PALETTE["light_blue"],
    },
    {
        "y": 0.550,
        "title": "Local graph",
        "body": (
            "Neighbour radius ≤8 Å; maximum 12 neighbours; "
            "distance expanded into 80 RBF bins"
        ),
        "edge": PALETTE["green"],
        "face": PALETTE["light_green"],
    },
    {
        "y": 0.400,
        "title": "Line graph",
        "body": (
            "Local edges become nodes; angle cosine in [−1, 1] "
            "expanded into 40 RBF bins"
        ),
        "edge": PALETTE["orange"],
        "face": PALETTE["light_orange"],
    },
    {
        "y": 0.250,
        "title": "Wider graph",
        "body": (
            "All non-self atom pairs; Coulomb interaction "
            "Cᵢⱼ = ZᵢZⱼ/rᵢⱼ"
        ),
        "edge": PALETTE["purple"],
        "face": PALETTE["light_purple"],
    },
]

input_centres = []

for specification in input_specs:

    box(
        ax,
        INPUT_X,
        specification["y"],
        INPUT_WIDTH,
        INPUT_HEIGHT,
        specification["title"],
        specification["body"],
        face=specification["face"],
        edge=specification["edge"],
        wrap=39,
        title_size=11.7,
        body_size=9.0,
    )

    input_centre_y = (
        specification["y"]
        + INPUT_HEIGHT / 2
    )

    input_centres.append(
        input_centre_y
    )

    # Horizontal line from each representation to the shared input bus.
    straight_line(
        (
            INPUT_X + INPUT_WIDTH,
            input_centre_y,
        ),
        (
            INPUT_BUS_X,
            input_centre_y,
        ),
        colour=specification["edge"],
        linewidth=1.35,
    )

    # Explicit coloured junction marker.
    ax.scatter(
        [INPUT_BUS_X],
        [input_centre_y],
        s=31,
        color=specification["edge"],
        edgecolor="white",
        linewidth=0.7,
        transform=ax.transAxes,
        clip_on=False,
        zorder=4,
    )


# -------------------------------------------------------------------------
# Shared input bus
# -------------------------------------------------------------------------

encoder_input_centre_y = (
    ENCODER_INPUT_Y
    + ENCODER_INPUT_HEIGHT / 2
)

# Vertical bus collecting all four representations.
straight_line(
    (
        INPUT_BUS_X,
        min(input_centres),
    ),
    (
        INPUT_BUS_X,
        encoder_input_centre_y,
    ),
    colour=CONNECTOR_COLOUR,
    linewidth=1.7,
)

# Straight arrow from the input bus to the embedded graph tensors.
straight_arrow(
    (
        INPUT_BUS_X,
        encoder_input_centre_y,
    ),
    (
        ENCODER_X,
        encoder_input_centre_y,
    ),
    colour=CONNECTOR_COLOUR,
)


# =========================================================================
# Panel B: complete MGT encoder block
# =========================================================================

# -------------------------------------------------------------------------
# Embedded graph tensors
# -------------------------------------------------------------------------

box(
    ax,
    ENCODER_X,
    ENCODER_INPUT_Y,
    ENCODER_WIDTH,
    ENCODER_INPUT_HEIGHT,
    "Embedded graph tensors",
    (
        "Atom, local-edge, line-graph and wider-edge features "
        "projected into the shared hidden space"
    ),
    face="#eef1f5",
    edge=PALETTE["navy"],
    wrap=55,
    title_size=10.5,
    body_size=8.0,
    lw=1.1,
)


# -------------------------------------------------------------------------
# Encoder stages
# -------------------------------------------------------------------------

# List encoder operations in the order shown by the adapted architecture.
encoder_steps = [
    {
        "y": 0.630,
        "title": "1. Wider-graph attention",
        "body": (
            "Coulomb wider graph; one multi-head layer; "
            "four attention heads; dropout 0.2"
        ),
        "edge": PALETTE["purple"],
        "face": PALETTE["light_purple"],
    },
    {
        "y": 0.510,
        "title": "2. Angular message passing",
        "body": (
            "Local and line graphs; three ALIGNN "
            "angle → edge → atom updates"
        ),
        "edge": PALETTE["orange"],
        "face": PALETTE["light_orange"],
    },
    {
        "y": 0.390,
        "title": "3. Local refinement",
        "body": (
            "Local graph; three post-ALIGNN "
            "edge-gated graph convolutions"
        ),
        "edge": PALETTE["green"],
        "face": PALETTE["light_green"],
    },
    {
        "y": 0.270,
        "title": "4. Atom-wise output block",
        "body": (
            "512 → 512 → 512 feed-forward block; "
            "LayerNorm and additive residual"
        ),
        "edge": PALETTE["blue"],
        "face": PALETTE["light_blue"],
    },
]

for step in encoder_steps:

    box(
        ax,
        ENCODER_X,
        step["y"],
        ENCODER_WIDTH,
        ENCODER_STEP_HEIGHT,
        step["title"],
        step["body"],
        face=step["face"],
        edge=step["edge"],
        wrap=52,
        title_size=10.8,
        body_size=8.5,
        lw=1.2,
    )


# -------------------------------------------------------------------------
# Sequential encoder arrows
# -------------------------------------------------------------------------

encoder_centre_x = (
    ENCODER_X
    + ENCODER_WIDTH / 2
)

# Embedded tensors to wider-graph attention.
straight_arrow(
    (
        encoder_centre_x,
        ENCODER_INPUT_Y,
    ),
    (
        encoder_centre_x,
        encoder_steps[0]["y"]
        + ENCODER_STEP_HEIGHT,
    ),
)

# Remaining encoder transitions.
for upper_step, lower_step in zip(
    encoder_steps[:-1],
    encoder_steps[1:],
):

    straight_arrow(
        (
            encoder_centre_x,
            upper_step["y"],
        ),
        (
            encoder_centre_x,
            lower_step["y"]
            + ENCODER_STEP_HEIGHT,
        ),
    )


# =========================================================================
# Panel C: raised and row-aligned ligand-level regression
# =========================================================================

# -------------------------------------------------------------------------
# Derive Panel C positions from Panel B
# -------------------------------------------------------------------------

# Global pooling aligns with embedded graph tensors.
pooling_centre_y = (
    ENCODER_INPUT_Y
    + ENCODER_INPUT_HEIGHT / 2
)

# Regression head aligns with wider-graph attention.
regression_centre_y = (
    encoder_steps[0]["y"]
    + ENCODER_STEP_HEIGHT / 2
)

# Inverse transformation aligns with angular message passing.
inverse_centre_y = (
    encoder_steps[1]["y"]
    + ENCODER_STEP_HEIGHT / 2
)

POOLING_Y = (
    pooling_centre_y
    - OUTPUT_BOX_HEIGHT / 2
)

REGRESSION_Y = (
    regression_centre_y
    - OUTPUT_BOX_HEIGHT / 2
)

INVERSE_Y = (
    inverse_centre_y
    - OUTPUT_BOX_HEIGHT / 2
)


# -------------------------------------------------------------------------
# Global mean pooling
# -------------------------------------------------------------------------

box(
    ax,
    OUTPUT_X,
    POOLING_Y,
    OUTPUT_WIDTH,
    OUTPUT_BOX_HEIGHT,
    "Global mean pooling",
    "Mean-pool contextual atom states into one ligand vector",
    face=PALETTE["light_teal"],
    edge=PALETTE["teal"],
    wrap=27,
    title_size=10.2,
    body_size=8.0,
)


# -------------------------------------------------------------------------
# Regression head
# -------------------------------------------------------------------------

box(
    ax,
    OUTPUT_X,
    REGRESSION_Y,
    OUTPUT_WIDTH,
    OUTPUT_BOX_HEIGHT,
    "Regression head",
    "Linear 512 → 1; predict normalised pKD",
    face=PALETTE["light_blue"],
    edge=PALETTE["blue"],
    wrap=27,
    title_size=10.2,
    body_size=8.0,
)


# -------------------------------------------------------------------------
# Inverse transformation
# -------------------------------------------------------------------------

box(
    ax,
    OUTPUT_X,
    INVERSE_Y,
    OUTPUT_WIDTH,
    OUTPUT_BOX_HEIGHT,
    "Inverse transform",
    "pKD = z·σtrain + μtrain",
    face=PALETTE["light_green"],
    edge=PALETTE["green"],
    wrap=27,
    title_size=10.2,
    body_size=8.0,
)


# -------------------------------------------------------------------------
# Route final encoder output to raised pooling box
# -------------------------------------------------------------------------

final_encoder_centre_y = (
    encoder_steps[-1]["y"]
    + ENCODER_STEP_HEIGHT / 2
)

# Dedicated routing lane between Panels B and C.
# Use the gap between encoder and readout columns for the vertical connector.
routing_lane_x = (
    ENCODER_X
    + ENCODER_WIDTH
    + (
        OUTPUT_X
        - (
            ENCODER_X
            + ENCODER_WIDTH
        )
    ) / 2
)

# Leave the final encoder block horizontally.
straight_line(
    (
        ENCODER_X + ENCODER_WIDTH,
        final_encoder_centre_y,
    ),
    (
        routing_lane_x,
        final_encoder_centre_y,
    ),
    colour=CONNECTOR_COLOUR,
    linewidth=1.6,
)

# Move vertically upward in the empty inter-panel lane.
straight_line(
    (
        routing_lane_x,
        final_encoder_centre_y,
    ),
    (
        routing_lane_x,
        pooling_centre_y,
    ),
    colour=CONNECTOR_COLOUR,
    linewidth=1.6,
)

# Enter global pooling horizontally.
straight_arrow(
    (
        routing_lane_x,
        pooling_centre_y,
    ),
    (
        OUTPUT_X,
        pooling_centre_y,
    ),
    colour=CONNECTOR_COLOUR,
    linewidth=1.6,
)


# -------------------------------------------------------------------------
# Straight regression pathway
# -------------------------------------------------------------------------

output_centre_x = (
    OUTPUT_X
    + OUTPUT_WIDTH / 2
)

# Pooling to regression.
straight_arrow(
    (
        output_centre_x,
        POOLING_Y,
    ),
    (
        output_centre_x,
        REGRESSION_Y
        + OUTPUT_BOX_HEIGHT,
    ),
)

# Regression to inverse transformation.
straight_arrow(
    (
        output_centre_x,
        REGRESSION_Y,
    ),
    (
        output_centre_x,
        INVERSE_Y
        + OUTPUT_BOX_HEIGHT,
    ),
)


# =========================================================================
# Scope of the ligand-only adaptation
# =========================================================================

# This scope box uses only the unused lower area beneath Panels A and B.
box(
    ax,
    0.045,
    0.055,
    0.685,
    0.115,
    "Scope of the ligand-only adaptation",
    (
        "Non-periodic ligand input only: no lattice, periodic boundary "
        "conditions, protein coordinates or protein sequence. "
        "Crystallographic ligand geometry is preserved."
    ),
    face=PALETTE["light_red"],
    edge=PALETTE["red"],
    wrap=91,
    title_size=11.2,
    body_size=9.1,
)


# =========================================================================
# Layout and alignment checks
# =========================================================================

# Panel boundaries.
assert INPUT_X + INPUT_WIDTH < INPUT_BUS_X
assert INPUT_BUS_X < ENCODER_X
assert ENCODER_X + ENCODER_WIDTH < routing_lane_x
assert routing_lane_x < OUTPUT_X
assert OUTPUT_X + OUTPUT_WIDTH <= 1.0

# Input boxes remain separated.
for upper, lower in zip(
    input_specs[:-1],
    input_specs[1:],
):
    assert (
        lower["y"] + INPUT_HEIGHT
        < upper["y"]
    )

# Encoder boxes remain separated.
assert (
    encoder_steps[0]["y"]
    + ENCODER_STEP_HEIGHT
    < ENCODER_INPUT_Y
)

for upper, lower in zip(
    encoder_steps[:-1],
    encoder_steps[1:],
):
    assert (
        lower["y"] + ENCODER_STEP_HEIGHT
        < upper["y"]
    )

# Exact cross-panel row alignment.
assert abs(
    pooling_centre_y
    - encoder_input_centre_y
) < 1e-12

assert abs(
    regression_centre_y
    - (
        encoder_steps[0]["y"]
        + ENCODER_STEP_HEIGHT / 2
    )
) < 1e-12

assert abs(
    inverse_centre_y
    - (
        encoder_steps[1]["y"]
        + ENCODER_STEP_HEIGHT / 2
    )
) < 1e-12

# Panel C boxes remain separated.
assert (
    REGRESSION_Y + OUTPUT_BOX_HEIGHT
    < POOLING_Y
)

assert (
    INVERSE_Y + OUTPUT_BOX_HEIGHT
    < REGRESSION_Y
)

# Scope box remains below Panels A and B.
assert (
    0.055 + 0.115
    < min(
        input_specs[-1]["y"],
        encoder_steps[-1]["y"],
    )
)

# Scope box stops before Panel C.
assert (
    0.045 + 0.685
    < OUTPUT_X
)


# -------------------------------------------------------------------------
# Export
# -------------------------------------------------------------------------

fig.tight_layout(
    rect=(
        0.01,
        0.01,
        0.99,
        0.95,
    )
)

save_figure(
    fig,
    "figure_2_5_adapted_mgt_architecture",
)

plt.show()

**Suggested caption — Figure 2.5.** Adapted ligand-MGT architecture. Elemental and deterministic Laplacian positional features were combined with local-distance, angular line-graph and wider Coulomb graph representations. One encoder block applied wider-graph multi-head attention, three ALIGNN layers, three local edge-gated convolutions and a feed-forward residual-normalisation block before global mean pooling and scalar pKD regression.

### Code used: Compound-level 3D representation

I choose a retained compound with two structures and a ligand size close to 22 heavy atoms, then read one reference SDF without generating a new conformer. Distances, one illustrative angle and Coulomb-inspired pair descriptors are calculated from those coordinates.

The local-edge illustration uses MGT's 8 Å/12-neighbour settings; the controlled 3D GNN has different settings. RBF encodings and compound prediction aggregation are described schematically, not computed by this plotting cell.


In [ ]:
# This illustration reads experimental coordinates; it does not create a training graph cache.
# =============================================================================
# FIGURE 2.7
# Compound-level view of crystallographic 3D ligand representation
# =============================================================================





# -----------------------------------------------------------------------------
# 1. Locate the MGT project independently of the notebook working directory.
# -----------------------------------------------------------------------------

current_directory = Path.cwd().resolve()

candidate_roots = [
    current_directory,
    current_directory.parent,
    current_directory / "MGT",
    current_directory.parent / "MGT",
]

MGT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (
            path
            / "OpenBind_EV-A71_2A"
            / "experiment_a_ligand_mgt"
            / "curated"
            / "openbind_ligand_structures.csv"
        ).is_file()
    ),
    None,
)

if MGT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the MGT project from the current notebook directory."
    )

EXPERIMENT_ROOT = (
    MGT_ROOT
    / "OpenBind_EV-A71_2A"
    / "experiment_a_ligand_mgt"
)

SOURCE_DATASET_ROOT = (
    MGT_ROOT
    / "OpenBind_EV-A71_2A"
    / "OpenBind_EV-A71_2A"
)

CURATED_PATH = (
    EXPERIMENT_ROOT
    / "curated"
    / "openbind_ligand_structures.csv"
)

# Reuse the notebook output directory if already defined.
if "FIGURE_ROOT" in globals():
    figure_output_directory = Path(FIGURE_ROOT)
else:
    figure_output_directory = (
        MGT_ROOT
        / "output"
        / "dissertation_analysis"
        / "figures"
    )

figure_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------------------------------
# 2. Select a real repeated compound with a visually manageable atom count.
# -----------------------------------------------------------------------------

curated = pd.read_csv(
    CURATED_PATH,
    dtype={"official_compound_group_id": str},
)

curated["official_compound_group_id"] = (
    curated["official_compound_group_id"]
    .str
    .zfill(16)
)

# Count the crystallographic structures available for every official compound.
# Choose a repeated compound so the lower workflow can explain structure averaging.
compound_structure_counts = (
    curated
    .groupby("official_compound_group_id")
    .size()
)

# Prefer a compound represented by exactly two crystallographic structures.
two_structure_compounds = compound_structure_counts[
    compound_structure_counts == 2
].index

candidate_examples = curated[
    curated["official_compound_group_id"].isin(
        two_structure_compounds
    )
].copy()

# Prefer approximately 22 heavy atoms so the graph remains readable.
candidate_examples["visualisation_score"] = (
    candidate_examples["heavy_atom_count"]
    .astype(float)
    .sub(22)
    .abs()
)

candidate_examples = candidate_examples.sort_values(
    [
        "visualisation_score",
        "official_compound_group_id",
        "complex_name",
    ]
)

if candidate_examples.empty:
    raise RuntimeError(
        "No repeated compound could be selected from the curated dataset."
    )

selected_row = candidate_examples.iloc[0]

compound_id = selected_row["official_compound_group_id"]
complex_name = selected_row["complex_name"]
experimental_pkd = float(selected_row["experimental_pKD"])

compound_rows = curated[
    curated["official_compound_group_id"] == compound_id
].copy()

number_of_structures = len(compound_rows)

source_sdf_path = (
    SOURCE_DATASET_ROOT
    / selected_row["source_ligand_ref_sdf"]
)

if not source_sdf_path.is_file():
    raise FileNotFoundError(source_sdf_path)

supplier = Chem.SDMolSupplier(
    str(source_sdf_path),
    removeHs=False,
    sanitize=True,
)

molecule = next(
    (
        supplied_molecule
        for supplied_molecule in supplier
        if supplied_molecule is not None
    ),
    None,
)

if molecule is None:
    raise ValueError(
        f"RDKit could not read {source_sdf_path}"
    )

if molecule.GetNumConformers() != 1:
    raise ValueError(
        "The selected ligand must contain exactly one crystallographic conformer."
    )


# -----------------------------------------------------------------------------
# 3. Extract atom types, coordinates and chemical bonds.
# -----------------------------------------------------------------------------

conformer = molecule.GetConformer()

coordinates = np.asarray(
    [
        list(conformer.GetAtomPosition(atom_index))
        for atom_index in range(molecule.GetNumAtoms())
    ],
    dtype=float,
)

# Centre the coordinates for easier comparison between the four panels.
# Recentring translates the view without changing distances or angles.
coordinates = coordinates - coordinates.mean(
    axis=0,
    keepdims=True,
)

atomic_numbers = np.asarray(
    [
        atom.GetAtomicNum()
        for atom in molecule.GetAtoms()
    ],
    dtype=int,
)

element_symbols = [
    atom.GetSymbol()
    for atom in molecule.GetAtoms()
]

chemical_bond_pairs = {
    tuple(
        sorted(
            (
                bond.GetBeginAtomIdx(),
                bond.GetEndAtomIdx(),
            )
        )
    )
    for bond in molecule.GetBonds()
}


# -----------------------------------------------------------------------------
# 4. Reconstruct the adapted-MGT local graph.
#
# The implemented MGT uses:
#   - an 8 Å local neighbour radius;
#   - at most 12 neighbours for each source atom;
#   - directed graph edges.
#
# Directed edges are drawn once as undirected lines in the figure.
# -----------------------------------------------------------------------------

LOCAL_CUTOFF = 8.0
MAXIMUM_LOCAL_NEIGHBOURS = 12

# Compute all pairwise separations once, then select each atom's local neighbours.
distance_matrix = np.linalg.norm(
    coordinates[:, None, :]
    - coordinates[None, :, :],
    axis=2,
)

directed_local_edges = []

for source_atom in range(len(coordinates)):

    possible_destinations = np.where(
        (
            distance_matrix[source_atom] > 0
        )
        & (
            distance_matrix[source_atom]
            <= LOCAL_CUTOFF
        )
    )[0]

    ordered_destinations = possible_destinations[
        np.argsort(
            distance_matrix[
                source_atom,
                possible_destinations,
            ]
        )
    ]

    selected_destinations = ordered_destinations[
        :MAXIMUM_LOCAL_NEIGHBOURS
    ]

    directed_local_edges.extend(
        [
            (
                source_atom,
                int(destination_atom),
            )
            for destination_atom in selected_destinations
        ]
    )

# Draw directed connections once as undirected pairs to reduce visual clutter.
local_edge_pairs = sorted(
    {
        tuple(sorted(edge))
        for edge in directed_local_edges
        if edge[0] != edge[1]
    }
)

additional_spatial_pairs = [
    edge
    for edge in local_edge_pairs
    if edge not in chemical_bond_pairs
]


# -----------------------------------------------------------------------------
# 5. Select one representative angular triplet.
#
# The complete line graph contains all compatible local-edge pairs. The figure
# highlights one triplet so the angle representation remains readable.
# -----------------------------------------------------------------------------

chemical_adjacency = {
    atom_index: set()
    for atom_index in range(len(coordinates))
}

local_adjacency = {
    atom_index: set()
    for atom_index in range(len(coordinates))
}

for atom_a, atom_b in chemical_bond_pairs:
    chemical_adjacency[atom_a].add(atom_b)
    chemical_adjacency[atom_b].add(atom_a)

for atom_a, atom_b in local_edge_pairs:
    local_adjacency[atom_a].add(atom_b)
    local_adjacency[atom_b].add(atom_a)

# Select one readable triplet; this is not the complete model line graph.
angular_candidates = []

for centre_atom in range(len(coordinates)):

    # Prefer conventional bonded angles where possible.
    neighbours = sorted(
        chemical_adjacency[centre_atom]
    )

    if len(neighbours) < 2:
        neighbours = sorted(
            local_adjacency[centre_atom]
        )

    for first_atom, second_atom in combinations(
        neighbours,
        2,
    ):
        first_vector = (
            coordinates[first_atom]
            - coordinates[centre_atom]
        )
        second_vector = (
            coordinates[second_atom]
            - coordinates[centre_atom]
        )

        cosine_value = np.dot(
            first_vector,
            second_vector,
        ) / (
            np.linalg.norm(first_vector)
            * np.linalg.norm(second_vector)
        )

        cosine_value = float(
            np.clip(
                cosine_value,
                -1.0,
                1.0,
            )
        )

        angle_degrees = float(
            np.degrees(
                np.arccos(cosine_value)
            )
        )

        # Prefer a clearly visible non-collinear angle.
        selection_score = abs(
            angle_degrees - 110.0
        )

        angular_candidates.append(
            (
                selection_score,
                first_atom,
                centre_atom,
                second_atom,
                cosine_value,
                angle_degrees,
            )
        )

if not angular_candidates:
    raise RuntimeError(
        "No valid angular triplet was found."
    )

(
    _,
    angle_atom_a,
    angle_centre,
    angle_atom_b,
    selected_angle_cosine,
    selected_angle_degrees,
) = min(
    angular_candidates,
    key=lambda item: item[0],
)


# -----------------------------------------------------------------------------
# 6. Construct the wider Coulomb graph.
#
# The computational graph contains directed edges for every non-self atom pair.
# Each pair is drawn once here to avoid duplicating the same interaction.
# -----------------------------------------------------------------------------

# Use atomic-number products over distance as descriptors, not electrostatic energies.
coulomb_interactions = []

for atom_a, atom_b in combinations(
    range(len(coordinates)),
    2,
):
    separation = distance_matrix[
        atom_a,
        atom_b,
    ]

    coulomb_value = (
        atomic_numbers[atom_a]
        * atomic_numbers[atom_b]
        / separation
    )

    coulomb_interactions.append(
        (
            atom_a,
            atom_b,
            float(coulomb_value),
        )
    )

# Use atomic-number products over distance as descriptors, not electrostatic energies.
coulomb_interactions = sorted(
    coulomb_interactions,
    key=lambda item: item[2],
    reverse=True,
)

# Highlight only the strongest interactions. All pairs remain faintly visible.
strongest_coulomb_pairs = {
    (
        atom_a,
        atom_b,
    )
    for atom_a, atom_b, _ in coulomb_interactions[:12]
}


# -----------------------------------------------------------------------------
# 7. Visualisation settings.
# -----------------------------------------------------------------------------

element_colours = {
    1: "#f2f2f2",   # Hydrogen
    6: "#4d4d4d",   # Carbon
    7: "#3b6fd8",   # Nitrogen
    8: "#e45756",   # Oxygen
    9: "#8bd646",   # Fluorine
    15: "#d889d8",  # Phosphorus
    16: "#e0b000",  # Sulfur
    17: "#42a844",  # Chlorine
    35: "#9b5a3c",  # Bromine
    53: "#7b4ab5",  # Iodine
}

element_sizes = {
    1: 28,
    6: 82,
    7: 95,
    8: 100,
    9: 105,
    15: 120,
    16: 125,
    17: 130,
    35: 145,
    53: 160,
}

atom_colours = [
    element_colours.get(
        atomic_number,
        "#9d9d9d",
    )
    for atomic_number in atomic_numbers
]

atom_sizes = [
    element_sizes.get(
        atomic_number,
        100,
    )
    for atomic_number in atomic_numbers
]


# Draw the selected atom pair using the unmodified relative 3D geometry.
def draw_edge(
    axis,
    atom_a,
    atom_b,
    colour,
    linewidth=1.5,
    alpha=1.0,
    linestyle="-",
    zorder=1,
):
    """Draw one edge between two atoms."""

    points = coordinates[
        [atom_a, atom_b]
    ]

    axis.plot(
        points[:, 0],
        points[:, 1],
        points[:, 2],
        color=colour,
        linewidth=linewidth,
        alpha=alpha,
        linestyle=linestyle,
        zorder=zorder,
    )


# Element colours identify atoms; marker size is a visual convention, not a model feature.
def draw_atoms(
    axis,
    label_heteroatoms=False,
):
    """Draw atoms using element-specific colours and sizes."""

    axis.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
        coordinates[:, 2],
        s=atom_sizes,
        c=atom_colours,
        edgecolors="#20252b",
        linewidths=0.55,
        depthshade=True,
        zorder=5,
    )

    if label_heteroatoms:
        for atom_index, symbol in enumerate(
            element_symbols
        ):
            if symbol not in {"C", "H"}:
                axis.text(
                    coordinates[atom_index, 0],
                    coordinates[atom_index, 1],
                    coordinates[atom_index, 2],
                    f" {symbol}",
                    fontsize=8,
                    fontweight="bold",
                    color="#20252b",
                    zorder=6,
                )


# Use equal spatial scales and one camera view so graph panels can be compared directly.
def format_3d_axis(
    axis,
    title,
    subtitle,
):
    """Apply a common orientation and scale to every molecular panel."""

    coordinate_minimum = coordinates.min(
        axis=0
    )
    coordinate_maximum = coordinates.max(
        axis=0
    )

    coordinate_centre = (
        coordinate_minimum
        + coordinate_maximum
    ) / 2

    radius = (
        np.max(
            coordinate_maximum
            - coordinate_minimum
        )
        / 2
        * 1.22
    )

    radius = max(
        radius,
        1.0,
    )

    axis.set_xlim(
        coordinate_centre[0] - radius,
        coordinate_centre[0] + radius,
    )
    axis.set_ylim(
        coordinate_centre[1] - radius,
        coordinate_centre[1] + radius,
    )
    axis.set_zlim(
        coordinate_centre[2] - radius,
        coordinate_centre[2] + radius,
    )

    axis.set_box_aspect(
        (1, 1, 1)
    )

    axis.view_init(
        elev=20,
        azim=-58,
    )

    axis.set_proj_type("ortho")
    axis.set_axis_off()

    axis.set_title(
        title,
        fontsize=14,
        fontweight="bold",
        color="#1f3557",
        pad=8,
    )

    axis.text2D(
        0.5,
        0.955,
        subtitle,
        transform=axis.transAxes,
        ha="center",
        va="top",
        fontsize=9.5,
        color="#4f5964",
    )


# Place panel letters in screen coordinates so rotating a molecule does not move the label.
def panel_marker(
    axis,
    label,
):
    """Add a dissertation-style panel identifier."""

    axis.text2D(
        0.025,
        0.96,
        label,
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=13,
        fontweight="bold",
        color="white",
        bbox={
            "boxstyle": "round,pad=0.28",
            "facecolor": "#1f3557",
            "edgecolor": "none",
        },
    )


# -----------------------------------------------------------------------------
# 8. Create the four molecular representation panels.
# -----------------------------------------------------------------------------

figure = plt.figure(
    figsize=(20, 11),
    facecolor="white",
)

grid = GridSpec(
    2,
    4,
    figure=figure,
    height_ratios=[3.25, 1.30],
    hspace=0.10,
    wspace=0.02,
)

molecular_axes = [
    figure.add_subplot(
        grid[0, panel_index],
        projection="3d",
    )
    for panel_index in range(4)
]


# Panel A: crystallographic coordinates and chemical orientation.
axis = molecular_axes[0]

for atom_a, atom_b in chemical_bond_pairs:
    draw_edge(
        axis,
        atom_a,
        atom_b,
        colour="#555b61",
        linewidth=2.0,
        alpha=0.90,
        zorder=2,
    )

draw_atoms(
    axis,
    label_heteroatoms=True,
)

format_3d_axis(
    axis,
    "Crystallographic atom coordinates",
    "Elemental atom state + 10-dimensional Laplacian PE",
)

panel_marker(
    axis,
    "A",
)


# Panel B: local distance graph.
axis = molecular_axes[1]

# Additional spatial neighbours are shown first.
for atom_a, atom_b in additional_spatial_pairs:
    draw_edge(
        axis,
        atom_a,
        atom_b,
        colour="#2a9d8f",
        linewidth=1.15,
        alpha=0.40,
        linestyle="--",
        zorder=1,
    )

# Chemical bonds remain visible for molecular orientation.
for atom_a, atom_b in chemical_bond_pairs:
    draw_edge(
        axis,
        atom_a,
        atom_b,
        colour="#4f5964",
        linewidth=1.9,
        alpha=0.85,
        zorder=2,
    )

draw_atoms(axis)

format_3d_axis(
    axis,
    "Local distance graph",
    "r ≤ 8 Å; maximum 12 neighbours; distance → 80 RBF bins",
)

panel_marker(
    axis,
    "B",
)


# Panel C: line graph and one representative angular interaction.
axis = molecular_axes[2]

# Show the complete local network faintly.
for atom_a, atom_b in local_edge_pairs:
    draw_edge(
        axis,
        atom_a,
        atom_b,
        colour="#b7bdc5",
        linewidth=0.9,
        alpha=0.24,
        zorder=1,
    )

# Highlight the two local edges defining one angle.
draw_edge(
    axis,
    angle_atom_a,
    angle_centre,
    colour="#f28e2b",
    linewidth=3.6,
    alpha=0.95,
    zorder=3,
)

draw_edge(
    axis,
    angle_centre,
    angle_atom_b,
    colour="#f28e2b",
    linewidth=3.6,
    alpha=0.95,
    zorder=3,
)

# Local graph edges become nodes in the line graph.
first_edge_midpoint = (
    coordinates[angle_atom_a]
    + coordinates[angle_centre]
) / 2

second_edge_midpoint = (
    coordinates[angle_centre]
    + coordinates[angle_atom_b]
) / 2

axis.scatter(
    [
        first_edge_midpoint[0],
        second_edge_midpoint[0],
    ],
    [
        first_edge_midpoint[1],
        second_edge_midpoint[1],
    ],
    [
        first_edge_midpoint[2],
        second_edge_midpoint[2],
    ],
    marker="s",
    s=72,
    color="#f28e2b",
    edgecolor="white",
    linewidth=0.8,
    zorder=7,
)

# Connect the two edge-nodes to illustrate one line-graph edge.
axis.plot(
    [
        first_edge_midpoint[0],
        second_edge_midpoint[0],
    ],
    [
        first_edge_midpoint[1],
        second_edge_midpoint[1],
    ],
    [
        first_edge_midpoint[2],
        second_edge_midpoint[2],
    ],
    color="#c65f00",
    linewidth=2.2,
    linestyle=":",
    zorder=6,
)

label_position = (
    first_edge_midpoint
    + second_edge_midpoint
) / 2

axis.text(
    label_position[0],
    label_position[1],
    label_position[2],
    (
        f"  θ = {selected_angle_degrees:.1f}°\n"
        f"  cos θ = {selected_angle_cosine:.2f}"
    ),
    fontsize=8.5,
    color="#9a4700",
    fontweight="bold",
    zorder=8,
)

draw_atoms(axis)

format_3d_axis(
    axis,
    "Line graph and angular view",
    "Local edges → line-graph nodes; cosine → 40 RBF bins",
)

panel_marker(
    axis,
    "C",
)


# Panel D: full Coulomb wider graph.
axis = molecular_axes[3]

coulomb_values = np.asarray(
    [
        interaction[2]
        for interaction in coulomb_interactions
    ],
    dtype=float,
)

# Log scaling changes edge visibility only; it is not an encoder feature transform.
log_coulomb_values = np.log1p(
    coulomb_values
)

normalised_coulomb_values = (
    log_coulomb_values
    - log_coulomb_values.min()
) / (
    log_coulomb_values.max()
    - log_coulomb_values.min()
    + 1e-12
)

for (
    interaction,
    normalised_value,
) in zip(
    coulomb_interactions,
    normalised_coulomb_values,
):
    atom_a, atom_b, _ = interaction

    is_highlighted = (
        atom_a,
        atom_b,
    ) in strongest_coulomb_pairs

    draw_edge(
        axis,
        atom_a,
        atom_b,
        colour=(
            "#7b4ab5"
            if is_highlighted
            else "#9f8bb5"
        ),
        linewidth=(
            0.8
            + 1.8 * normalised_value
            if is_highlighted
            else 0.45
        ),
        alpha=(
            0.50
            if is_highlighted
            else 0.055
        ),
        zorder=(
            3
            if is_highlighted
            else 1
        ),
    )

draw_atoms(axis)

format_3d_axis(
    axis,
    "Wider Coulomb graph",
    r"All non-self atom pairs; $C_{ij}=Z_iZ_j/r_{ij}$",
)

panel_marker(
    axis,
    "D",
)


# -----------------------------------------------------------------------------
# 9. Add the compound-level aggregation workflow.
# -----------------------------------------------------------------------------

flow_axis = figure.add_subplot(
    grid[1, :]
)

flow_axis.set_xlim(
    0,
    1,
)

flow_axis.set_ylim(
    0,
    1,
)

flow_axis.axis("off")


# Display the compound-aggregation steps schematically; no predictions are calculated here.
def draw_flow_box(
    axis,
    x_position,
    width,
    title,
    body,
    face_colour,
    edge_colour,
):
    """Draw one stage of the compound-level aggregation workflow."""

    y_position = 0.34
    height = 0.40

    patch = FancyBboxPatch(
        (
            x_position,
            y_position,
        ),
        width,
        height,
        boxstyle="round,pad=0.012,rounding_size=0.018",
        facecolor=face_colour,
        edgecolor=edge_colour,
        linewidth=1.7,
    )

    axis.add_patch(patch)

    axis.text(
        x_position + width / 2,
        y_position + 0.275,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        color=edge_colour,
    )

    axis.text(
        x_position + width / 2,
        y_position + 0.135,
        body,
        ha="center",
        va="center",
        fontsize=9.2,
        color="#252a30",
        linespacing=1.25,
    )

    return (
        x_position,
        y_position,
        width,
        height,
    )


flow_boxes = [
    draw_flow_box(
        flow_axis,
        0.015,
        0.165,
        "Official compound group",
        (
            f"{compound_id}\n"
            f"{number_of_structures} retained crystal structures"
        ),
        "#dbe9f5",
        "#4c78a8",
    ),
    draw_flow_box(
        flow_axis,
        0.215,
        0.165,
        "Independent 3D encoding",
        (
            "Each crystallographic pose\n"
            "is processed separately through A–D"
        ),
        "#e1f0dd",
        "#59a14f",
    ),
    draw_flow_box(
        flow_axis,
        0.415,
        0.165,
        "Atom-level pooling",
        (
            "Mean-pool contextual atom states\n"
            "within each structure"
        ),
        "#d7efed",
        "#2a9d8f",
    ),
    draw_flow_box(
        flow_axis,
        0.615,
        0.165,
        "Structure predictions",
        (
            f"Regression head produces\n"
            f"{number_of_structures} structure-level pKD values"
        ),
        "#eee5f4",
        "#8e6bb7",
    ),
    draw_flow_box(
        flow_axis,
        0.815,
        0.165,
        "Compound-level prediction",
        (
            "Average repeated-pose predictions\n"
            "to one out-of-fold compound pKD"
        ),
        "#fff0df",
        "#f28e2b",
    ),
]

# Connect the workflow using straight horizontal arrows.
for first_box, second_box in zip(
    flow_boxes[:-1],
    flow_boxes[1:],
):
    first_x, first_y, first_width, first_height = first_box
    second_x, second_y, _, second_height = second_box

    flow_axis.annotate(
        "",
        xy=(
            second_x - 0.005,
            second_y + second_height / 2,
        ),
        xytext=(
            first_x + first_width + 0.005,
            first_y + first_height / 2,
        ),
        arrowprops={
            "arrowstyle": "-|>",
            "color": "#1f3557",
            "linewidth": 1.8,
            "mutation_scale": 14,
            "shrinkA": 0,
            "shrinkB": 0,
        },
    )


# Legend explaining the molecular panels.
legend_handles = [
    Line2D(
        [0],
        [0],
        color="#555b61",
        linewidth=2.2,
        label="Chemical bond shown for orientation",
    ),
    Line2D(
        [0],
        [0],
        color="#2a9d8f",
        linewidth=1.5,
        linestyle="--",
        label="Additional local spatial edge",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        color="#f28e2b",
        markerfacecolor="#f28e2b",
        linewidth=1.5,
        label="Line-graph edge-node",
    ),
    Line2D(
        [0],
        [0],
        color="#7b4ab5",
        linewidth=2.0,
        label="Highlighted Coulomb interaction",
    ),
]

flow_axis.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=4,
    frameon=False,
    fontsize=9.5,
)

flow_axis.text(
    0.5,
    0.11,
    (
        "Representation subsets: 3D distance GNN = A+B; "
        "3D ALIGNN = A+B+C; adapted MGT = A+B+C+D. "
        "Directed edges are drawn once for clarity."
    ),
    ha="center",
    va="center",
    fontsize=10.5,
    fontweight="bold",
    color="#1f3557",
)


# -----------------------------------------------------------------------------
# 10. Figure title, source note and export.
# -----------------------------------------------------------------------------

figure.suptitle(
    "How a crystallographic ligand is represented at compound level",
    fontsize=21,
    fontweight="bold",
    color="#20252b",
    y=0.985,
)

figure.text(
    0.5,
    0.935,
    (
        f"Example structure: {complex_name}  •  "
        f"official compound group: {compound_id}  •  "
        f"experimental pKD: {experimental_pkd:.3f}"
    ),
    ha="center",
    va="center",
    fontsize=11.5,
    color="#4f5964",
)

figure.text(
    0.5,
    0.906,
    (
        "The drawing uses the reference-SDF coordinates and bond identities. "
        "Adapted MGT receives the coordinate-equivalent ligand-only PDB."
    ),
    ha="center",
    va="center",
    fontsize=10,
    color="#6c737b",
    style="italic",
)

figure.subplots_adjust(
    left=0.018,
    right=0.982,
    top=0.875,
    bottom=0.035,
)

png_path = (
    figure_output_directory
    / "figure_2_7_compound_level_3d_model_view.png"
)

pdf_path = (
    figure_output_directory
    / "figure_2_7_compound_level_3d_model_view.pdf"
)

figure.savefig(
    png_path,
    dpi=350,
    bbox_inches="tight",
    facecolor="white",
)

figure.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print(f"Saved PNG: {png_path}")
print(f"Saved PDF: {pdf_path}")
print(f"Selected structure: {complex_name}")
print(f"Selected compound group: {compound_id}")
print(f"Retained structures in group: {number_of_structures}")

## Figure 2.6 — Matched training, masked pretraining and compound-level evaluation

**Recommended placement:** Sections 2.12–2.17, or the appendix if the chapter is figure-heavy.

### Code used: Supervised and masked-training pathways

I draw the unmasked and masking pathways separately and join them at validation-controlled selection. The connector starts at the top of the fine-tuning box; this cell illustrates the training procedure without running it.


In [ ]:
# Both routes share validation selection and untouched-test evaluation.
# -------------------------------------------------------------------------
# Straight connector helpers.
# -------------------------------------------------------------------------

def straight_line(
    ax,
    start,
    end,
    colour=None,
    linewidth=1.8,
    linestyle="-",
    zorder=4,
):
    """Draw a straight connector without an arrowhead."""

    if colour is None:
        colour = PALETTE["navy"]

    connector = Line2D(
        [start[0], end[0]],
        [start[1], end[1]],
        transform=ax.transAxes,
        color=colour,
        linewidth=linewidth,
        linestyle=linestyle,
        solid_capstyle="round",
        clip_on=False,
        zorder=zorder,
    )

    ax.add_line(connector)
    return connector


def straight_arrow(
    ax,
    start,
    end,
    colour=None,
    linewidth=1.8,
    mutation_scale=13,
    zorder=5,
):
    """Draw a straight connector with an arrowhead."""

    if colour is None:
        colour = PALETTE["navy"]

    connector = FancyArrowPatch(
        start,
        end,
        transform=ax.transAxes,
        arrowstyle="-|>",
        connectionstyle="arc3,rad=0",
        color=colour,
        linewidth=linewidth,
        mutation_scale=mutation_scale,
        shrinkA=0,
        shrinkB=0,
        clip_on=False,
        zorder=zorder,
    )

    ax.add_patch(connector)
    return connector


# -------------------------------------------------------------------------
# Create the figure.
# -------------------------------------------------------------------------

fig, ax = canvas(
    (18, 10),
    "Matched supervised training and masked-pretraining experiment",
)


# -------------------------------------------------------------------------
# Panel headings.
# -------------------------------------------------------------------------

panel_label(
    ax,
    0.035,
    0.92,
    "A",
    "Unmasked supervised pathway",
)

panel_label(
    ax,
    0.035,
    0.47,
    "B",
    "Masked atom-feature pathway",
)

panel_label(
    ax,
    0.705,
    0.92,
    "C",
    "Shared checkpointing and evaluation",
)


# =========================================================================
# A. UNMASKED SUPERVISED PATHWAY
# =========================================================================

box(
    ax,
    0.03,
    0.69,
    0.14,
    0.14,
    "Training fold",
    "Structures from training compound/scaffold groups only",
    face=PALETTE["light_blue"],
    edge=PALETTE["blue"],
    wrap=20,
)

box(
    ax,
    0.22,
    0.69,
    0.16,
    0.14,
    "Target scaling",
    "Training-fold μ and population σ; convert pKD to z",
    face=PALETTE["light_teal"],
    edge=PALETTE["teal"],
    wrap=22,
)

box(
    ax,
    0.43,
    0.69,
    0.19,
    0.14,
    "Supervised optimisation",
    "Huber δ=1; Adam 10⁻⁴; weight decay 10⁻⁵; batch 32",
    face=PALETTE["light_green"],
    edge=PALETTE["green"],
    wrap=27,
)

# Training fold → target scaling.
straight_arrow(
    ax,
    (0.172, 0.76),
    (0.217, 0.76),
)

# Target scaling → supervised optimisation.
straight_arrow(
    ax,
    (0.382, 0.76),
    (0.427, 0.76),
)


# =========================================================================
# B. MASKED ATOM-FEATURE PATHWAY
# =========================================================================

box(
    ax,
    0.03,
    0.24,
    0.14,
    0.14,
    "Training fold",
    "No validation or test structures used for pretraining",
    face=PALETTE["light_blue"],
    edge=PALETTE["blue"],
    wrap=20,
)

box(
    ax,
    0.205,
    0.24,
    0.15,
    0.14,
    "Mask features",
    "Zero complete input vectors for ≈20% of atoms",
    face=PALETTE["light_orange"],
    edge=PALETTE["orange"],
    wrap=21,
)

box(
    ax,
    0.39,
    0.24,
    0.15,
    0.14,
    "Reconstruct",
    "Encoder 512-D states → linear decoder; masked-node MSE; 30 epochs",
    face=PALETTE["light_purple"],
    edge=PALETTE["purple"],
    wrap=23,
)

box(
    ax,
    0.575,
    0.24,
    0.12,
    0.14,
    "Fine-tune",
    "Discard decoder; update all encoder weights for pKD",
    face=PALETTE["light_green"],
    edge=PALETTE["green"],
    wrap=18,
)

# Training fold → mask features.
straight_arrow(
    ax,
    (0.172, 0.31),
    (0.202, 0.31),
)

# Mask features → reconstruction.
straight_arrow(
    ax,
    (0.357, 0.31),
    (0.387, 0.31),
)

# Reconstruction → fine-tuning.
straight_arrow(
    ax,
    (0.542, 0.31),
    (0.572, 0.31),
)


# -------------------------------------------------------------------------
# Masking clarification.
# -------------------------------------------------------------------------

ax.text(
    0.36,
    0.17,
    (
        "Connectivity, distances, angles and MGT Coulomb edges remain visible: "
        "atom-feature masking, not total atom removal"
    ),
    ha="center",
    va="center",
    fontsize=10.5,
    color=PALETTE["red"],
    fontweight="bold",
)


# =========================================================================
# MERGE THE MASKED AND UNMASKED PATHWAYS
# =========================================================================

# Centre height of the upper supervised pathway.
# The masking route rises from the fine-tuning box to the supervised route's centreline.
top_path_y = 0.76

# Fine-tune box:
# left = 0.575
# bottom = 0.24
# width = 0.12
# height = 0.14
#
# Therefore, its top-centre coordinate is:
fine_tune_top_x = 0.575 + (0.12 / 2)
fine_tune_top_y = 0.24 + 0.14

# Continue the unmasked pathway from supervised optimisation towards the
# validation-controlled selection box.
straight_arrow(
    ax,
    (0.622, top_path_y),
    (0.717, top_path_y),
)

# The masked pathway leaves from the TOP CENTRE of the Fine-tune box.
# It travels vertically upward and joins the upper pathway.
straight_line(
    ax,
    (fine_tune_top_x, fine_tune_top_y),
    (fine_tune_top_x, top_path_y),
)

# Mark the exact point at which both pathways merge.
ax.scatter(
    [fine_tune_top_x],
    [top_path_y],
    transform=ax.transAxes,
    s=55,
    color=PALETTE["navy"],
    edgecolor="white",
    linewidth=0.9,
    zorder=7,
    clip_on=False,
)


# =========================================================================
# C. SHARED CHECKPOINTING AND EVALUATION
# =========================================================================

box(
    ax,
    0.72,
    0.69,
    0.24,
    0.14,
    "Validation-controlled selection",
    (
        "ReduceLROnPlateau: factor 0.5, patience 5; "
        "early stopping patience 10; restore minimum "
        "validation-Huber checkpoint"
    ),
    face="#fff7df",
    edge="#c79a1b",
    wrap=35,
    body_size=9.5,
)

box(
    ax,
    0.72,
    0.46,
    0.24,
    0.14,
    "Untouched outer test fold",
    (
        "Predict only after checkpoint selection; "
        "transform z back to original pKD units"
    ),
    face=PALETTE["light_red"],
    edge=PALETTE["red"],
    wrap=34,
    linestyle="dashed",
)

# Checkpoint selection → untouched outer test fold.
straight_arrow(
    ax,
    (0.84, 0.685),
    (0.84, 0.605),
    colour=PALETTE["red"],
)

box(
    ax,
    0.72,
    0.23,
    0.24,
    0.14,
    "Compound-level evaluation",
    (
        "Average predictions across repeated crystal structures; "
        "pool one OOF prediction per compound"
    ),
    face=PALETTE["light_teal"],
    edge=PALETTE["teal"],
    wrap=35,
)

# Outer test fold → compound-level evaluation.
straight_arrow(
    ax,
    (0.84, 0.455),
    (0.84, 0.375),
)

box(
    ax,
    0.72,
    0.055,
    0.24,
    0.105,
    "Reported metrics",
    "MAE • MSE • RMSE • R² • Pearson • Spearman",
    face=PALETTE["light_grey"],
    edge=PALETTE["navy"],
    wrap=34,
    title_size=11.5,
    body_size=9.5,
)

# Compound-level evaluation → reported metrics.
straight_arrow(
    ax,
    (0.84, 0.225),
    (0.84, 0.165),
)


# -------------------------------------------------------------------------
# Experimental summary.
# -------------------------------------------------------------------------

ax.text(
    0.36,
    0.055,
    (
        "Maximum 200 epochs • shared optimiser and evaluation • "
        "70 supervised fits = 7 configurations × 2 CV designs × 5 folds"
    ),
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
    color=PALETTE["navy"],
)


# -------------------------------------------------------------------------
# Final layout and export.
# -------------------------------------------------------------------------

fig.tight_layout(
    rect=(0.01, 0.01, 0.99, 0.95),
)

save_figure(
    fig,
    "figure_2_6_training_masking_evaluation",
)

**Suggested caption — Figure 2.6.** Matched supervised and masked-pretraining pathways. Target normalisation and atom-feature reconstruction were fitted using the training fold only. Validation loss controlled scheduling, early stopping and checkpoint selection; the outer test fold was predicted once using the restored checkpoint. Repeated-structure predictions were averaged before compound-level out-of-fold metrics were calculated.

### Code used: Appendix architecture figures

I reuse one three-column template for the six additional model diagrams. Each configuration defines its input branches, encoder blocks and output stages, so the comparison keeps a consistent layout without duplicating the drawing code.


In [ ]:
# Model-specific content is separated from the reusable appendix layout.
# =========================================================================
# STRAIGHT CONNECTOR HELPERS
# =========================================================================

def architecture_line(
    ax,
    start,
    end,
    colour=None,
    linewidth=1.6,
    linestyle="-",
    zorder=3,
):
    """Draw one straight line in axes-relative coordinates."""

    if colour is None:
        colour = PALETTE["navy"]

    line = Line2D(
        [start[0], end[0]],
        [start[1], end[1]],
        transform=ax.transAxes,
        color=colour,
        linewidth=linewidth,
        linestyle=linestyle,
        solid_capstyle="round",
        clip_on=False,
        zorder=zorder,
    )

    ax.add_line(line)
    return line


def architecture_arrow(
    ax,
    start,
    end,
    colour=None,
    linewidth=1.6,
    mutation_scale=12,
    zorder=4,
):
    """Draw one straight arrow in axes-relative coordinates."""

    if colour is None:
        colour = PALETTE["navy"]

    connector = FancyArrowPatch(
        start,
        end,
        transform=ax.transAxes,
        arrowstyle="-|>",
        connectionstyle="arc3,rad=0",
        color=colour,
        linewidth=linewidth,
        mutation_scale=mutation_scale,
        shrinkA=0,
        shrinkB=0,
        clip_on=False,
        zorder=zorder,
    )

    ax.add_patch(connector)
    return connector


# Allow each model specification to use either the shared palette or an explicit colour.
def architecture_colour(value):
    """Resolve a colour from PALETTE or return a literal colour."""

    return PALETTE[value] if value in PALETTE else value


# =========================================================================
# FIXED, NON-OVERLAPPING LAYOUT
# =========================================================================

PANEL_A_X = 0.030
PANEL_A_WIDTH = 0.270

PANEL_B_X = 0.390
PANEL_B_WIDTH = 0.340

PANEL_C_X = 0.800
PANEL_C_WIDTH = 0.170

INPUT_BUS_X = 0.340
OUTPUT_ROUTE_X = 0.765


# Choose fixed layouts by block count to maintain alignment across appendix models.
def stack_positions(number_of_boxes):
    """Return top-to-bottom box positions for panels A and B."""

    layouts = {
        1: ([0.470], 0.130),
        2: ([0.635, 0.300], 0.120),
        3: ([0.665, 0.440, 0.215], 0.115),
        4: ([0.690, 0.530, 0.370, 0.210], 0.110),
        5: ([0.705, 0.580, 0.455, 0.330, 0.205], 0.100),
    }

    if number_of_boxes not in layouts:
        raise ValueError(
            f"No layout has been defined for {number_of_boxes} boxes."
        )

    return layouts[number_of_boxes]


# Reserve a separate vertical layout for pooling and prediction blocks.
def output_positions(number_of_boxes):
    """Return top-to-bottom positions for the regression panel."""

    layouts = {
        2: ([0.585, 0.265], 0.140),
        3: ([0.620, 0.370, 0.120], 0.135),
    }

    if number_of_boxes not in layouts:
        raise ValueError(
            f"No output layout has been defined for {number_of_boxes} boxes."
        )

    return layouts[number_of_boxes]


# Return each box's boundaries so later connectors can be routed without overlaps.
def draw_architecture_stack(
    ax,
    specifications,
    x,
    width,
    positions,
    height,
):
    """Draw an aligned vertical stack and return its box geometry."""

    geometries = []

    for specification, y in zip(specifications, positions):

        edge_colour = architecture_colour(specification["edge"])
        face_colour = architecture_colour(specification["face"])

        box(
            ax,
            x,
            y,
            width,
            height,
            specification["title"],
            specification["body"],
            face=face_colour,
            edge=edge_colour,
            wrap=specification.get("wrap", 35),
            title_size=specification.get("title_size", 11.5),
            body_size=specification.get("body_size", 9.2),
            lw=specification.get("linewidth", 1.5),
            linestyle=specification.get("linestyle", "-"),
        )

        geometries.append(
            {
                "x": x,
                "y": y,
                "width": width,
                "height": height,
                "left": x,
                "right": x + width,
                "bottom": y,
                "top": y + height,
                "centre_x": x + width / 2,
                "centre_y": y + height / 2,
                "edge": edge_colour,
            }
        )

    return geometries


# Link consecutive processing stages from each lower edge to the next upper edge.
def connect_vertical_stack(ax, geometries):
    """Connect a top-to-bottom stack using short straight arrows."""

    for upper, lower in zip(geometries[:-1], geometries[1:]):

        architecture_arrow(
            ax,
            (
                upper["centre_x"],
                upper["bottom"] - 0.003,
            ),
            (
                lower["centre_x"],
                lower["top"] + 0.003,
            ),
        )


# Merge simultaneous feature views through a shared routing line.
def connect_parallel_inputs(ax, input_boxes, first_encoder_box):
    """Merge several parallel feature inputs through a straight bus."""

    target_y = first_encoder_box["centre_y"]

    all_y_positions = [
        geometry["centre_y"]
        for geometry in input_boxes
    ] + [target_y]

    # Connect each input box to the common vertical bus.
    for geometry in input_boxes:

        architecture_line(
            ax,
            (
                geometry["right"],
                geometry["centre_y"],
            ),
            (
                INPUT_BUS_X,
                geometry["centre_y"],
            ),
            colour=geometry["edge"],
        )

        ax.scatter(
            [INPUT_BUS_X],
            [geometry["centre_y"]],
            transform=ax.transAxes,
            s=25,
            color=geometry["edge"],
            edgecolor="white",
            linewidth=0.6,
            zorder=5,
            clip_on=False,
        )

    # Draw the vertical bus.
    architecture_line(
        ax,
        (
            INPUT_BUS_X,
            min(all_y_positions),
        ),
        (
            INPUT_BUS_X,
            max(all_y_positions),
        ),
    )

    # Feed the merged representation into the encoder.
    architecture_arrow(
        ax,
        (
            INPUT_BUS_X,
            target_y,
        ),
        (
            first_encoder_box["left"] - 0.004,
            target_y,
        ),
    )


# Preserve the order of preprocessing stages before routing into the encoder.
def connect_sequential_inputs(ax, input_boxes, first_encoder_box):
    """Connect sequential input-processing boxes to the encoder."""

    connect_vertical_stack(
        ax,
        input_boxes,
    )

    final_input = input_boxes[-1]
    target_y = first_encoder_box["centre_y"]

    # Move right from the final input stage.
    architecture_line(
        ax,
        (
            final_input["right"],
            final_input["centre_y"],
        ),
        (
            INPUT_BUS_X,
            final_input["centre_y"],
        ),
        colour=final_input["edge"],
    )

    # Move vertically to the encoder entrance.
    architecture_line(
        ax,
        (
            INPUT_BUS_X,
            final_input["centre_y"],
        ),
        (
            INPUT_BUS_X,
            target_y,
        ),
    )

    # Enter the encoder.
    architecture_arrow(
        ax,
        (
            INPUT_BUS_X,
            target_y,
        ),
        (
            first_encoder_box["left"] - 0.004,
            target_y,
        ),
    )


# Route the last encoder stage around the boxes into the ligand readout.
def connect_encoder_to_output(
    ax,
    final_encoder_box,
    first_output_box,
):
    """Route the encoder output orthogonally into ligand pooling/output."""

    source_y = final_encoder_box["centre_y"]
    target_y = first_output_box["centre_y"]

    # Leave the final encoder stage horizontally.
    architecture_line(
        ax,
        (
            final_encoder_box["right"],
            source_y,
        ),
        (
            OUTPUT_ROUTE_X,
            source_y,
        ),
    )

    # Move vertically in the empty gap between panels B and C.
    architecture_line(
        ax,
        (
            OUTPUT_ROUTE_X,
            source_y,
        ),
        (
            OUTPUT_ROUTE_X,
            target_y,
        ),
    )

    # Enter the first output box.
    architecture_arrow(
        ax,
        (
            OUTPUT_ROUTE_X,
            target_y,
        ),
        (
            first_output_box["left"] - 0.004,
            target_y,
        ),
    )


# =========================================================================
# ARCHITECTURE CONFIGURATIONS
# =========================================================================

# The dictionaries contain the hand-specified architecture content for each appendix figure.
ARCHITECTURE_CONFIGURATIONS = {
    "morgan_mlp": {
        "title": "Morgan fingerprint multilayer perceptron",
        "panel_a": "Fixed molecular representation",
        "panel_b": "Feed-forward predictor",
        "panel_c": "Affinity regression",
        "input_mode": "sequence",
        "inputs": [
            {
                "title": "Canonical SMILES",
                "body": (
                    "RDKit-parsed canonical molecular identity; "
                    "no coordinates or graph message passing"
                ),
                "edge": "grey",
                "face": "light_grey",
                "wrap": 34,
            },
            {
                "title": "Morgan fingerprint",
                "body": (
                    "Radius 2; 2,048-bit binary circular fingerprint"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 32,
            },
        ],
        "encoder": [
            {
                "title": "Hidden layer 1",
                "body": (
                    "Linear 2,048 → 128; BatchNorm; ReLU; dropout 0.4"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 39,
            },
            {
                "title": "Hidden layer 2",
                "body": (
                    "Linear 128 → 64; BatchNorm; ReLU; dropout 0.4"
                ),
                "edge": "teal",
                "face": "light_teal",
                "wrap": 39,
            },
        ],
        "output": [
            {
                "title": "Regression head",
                "body": "Linear 64 → 1; predict normalised pKD",
                "edge": "blue",
                "face": "light_blue",
                "wrap": 25,
            },
            {
                "title": "Inverse transform",
                "body": "pKD = z·σtrain + μtrain",
                "edge": "green",
                "face": "light_green",
                "wrap": 25,
            },
        ],
        "footer_title": "Controlled baseline",
        "footer_body": (
            "270,977 trainable parameters. Tests whether a fixed 2D "
            "fingerprint can predict affinity without learned molecular graphs."
        ),
        "filename": "appendix_architecture_morgan_mlp",
    },

    "2d_gnn": {
        "title": "Two-dimensional ligand graph neural network",
        "panel_a": "Chemical graph representation",
        "panel_b": "Edge-gated graph encoder",
        "panel_c": "Ligand-level regression",
        "input_mode": "parallel",
        "inputs": [
            {
                "title": "Atom features",
                "body": "152-dimensional RDKit atom descriptors",
                "edge": "blue",
                "face": "light_blue",
                "wrap": 30,
            },
            {
                "title": "Bond features",
                "body": "12-dimensional RDKit chemical-bond descriptors",
                "edge": "teal",
                "face": "light_teal",
                "wrap": 30,
            },
            {
                "title": "Directed chemical graph",
                "body": (
                    "Atoms are nodes; chemical bonds are directed edges; "
                    "no coordinates or spatial neighbours"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 35,
            },
        ],
        "encoder": [
            {
                "title": "Feature projection",
                "body": (
                    "Atom features 152 → 512; bond features "
                    "12 → 128 → 512"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 40,
            },
            {
                "title": "Chemical message passing",
                "body": (
                    "Three EdgeGatedGraphConv layers with hidden width 512"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 42,
            },
        ],
        "output": [
            {
                "title": "Global mean pooling",
                "body": "Mean-pool contextual atom states into one ligand vector",
                "edge": "teal",
                "face": "light_teal",
                "wrap": 25,
            },
            {
                "title": "Regression head",
                "body": "Linear 512 → 1; predict normalised pKD",
                "edge": "blue",
                "face": "light_blue",
                "wrap": 25,
            },
            {
                "title": "Inverse transform",
                "body": "pKD = z·σtrain + μtrain",
                "edge": "green",
                "face": "light_green",
                "wrap": 25,
            },
        ],
        "footer_title": "Controlled experimental role",
        "footer_body": (
            "4,094,849 trainable parameters. Tests learned atom–bond "
            "representations without crystallographic coordinates."
        ),
        "filename": "appendix_architecture_2d_gnn",
    },

    "3d_distance_gnn": {
        "title": "Crystallographic three-dimensional distance GNN",
        "panel_a": "Chemical and spatial graph construction",
        "panel_b": "Distance-aware graph encoder",
        "panel_c": "Ligand-level regression",
        "input_mode": "parallel",
        "inputs": [
            {
                "title": "Atom and bond chemistry",
                "body": (
                    "152-dimensional atom features and "
                    "12-dimensional bond features"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 34,
            },
            {
                "title": "Crystal coordinates",
                "body": (
                    "Experimental ligand_ref.sdf coordinates; "
                    "no conformer generation or optimisation"
                ),
                "edge": "teal",
                "face": "light_teal",
                "wrap": 35,
            },
            {
                "title": "Spatial graph",
                "body": (
                    "Union of chemical bonds and neighbours ≤5 Å; "
                    "maximum 32 spatial neighbours per source atom"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 37,
            },
            {
                "title": "Distance features",
                "body": (
                    "Interatomic distance expanded from 0–5 Å "
                    "into 40 Gaussian RBF bins"
                ),
                "edge": "orange",
                "face": "light_orange",
                "wrap": 36,
            },
        ],
        "encoder": [
            {
                "title": "Feature embedding",
                "body": (
                    "Atoms 152 → 512; edge chemistry plus distance RBF "
                    "52 → 128 → 512"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 45,
            },
            {
                "title": "Distance-aware message passing",
                "body": (
                    "Three EdgeGatedGraphConv layers with hidden width 512"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 44,
            },
        ],
        "output": [
            {
                "title": "Global mean pooling",
                "body": "Mean-pool contextual atom states into one ligand vector",
                "edge": "teal",
                "face": "light_teal",
                "wrap": 25,
            },
            {
                "title": "Regression head",
                "body": "Linear 512 → 1; predict normalised pKD",
                "edge": "blue",
                "face": "light_blue",
                "wrap": 25,
            },
            {
                "title": "Inverse transform",
                "body": "pKD = z·σtrain + μtrain",
                "edge": "green",
                "face": "light_green",
                "wrap": 25,
            },
        ],
        "footer_title": "Controlled experimental role",
        "footer_body": (
            "4,099,969 trainable parameters. Comparison with the 2D GNN "
            "isolates the value of crystallographic spatial edges and distances."
        ),
        "filename": "appendix_architecture_3d_distance_gnn",
    },

    "3d_alignn": {
        "title": "Crystallographic three-dimensional ALIGNN",
        "panel_a": "Local graph and angular representation",
        "panel_b": "ALIGNN encoder",
        "panel_c": "Ligand-level regression",
        "input_mode": "parallel",
        "inputs": [
            {
                "title": "Atom and bond chemistry",
                "body": (
                    "152-dimensional atom features and "
                    "12-dimensional chemical-bond features"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 35,
            },
            {
                "title": "Crystal spatial graph",
                "body": (
                    "Chemical bonds plus spatial neighbours ≤5 Å; "
                    "maximum 32 neighbours; 40-bin distance RBF"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 37,
            },
            {
                "title": "Line graph",
                "body": (
                    "Local graph edges become line-graph nodes; "
                    "connected edge pairs define bond angles"
                ),
                "edge": "orange",
                "face": "light_orange",
                "wrap": 36,
            },
            {
                "title": "Angular features",
                "body": (
                    "Angle cosine in [−1, 1] expanded into "
                    "40 Gaussian RBF bins"
                ),
                "edge": "purple",
                "face": "light_purple",
                "wrap": 35,
            },
        ],
        "encoder": [
            {
                "title": "Feature embedding",
                "body": (
                    "Atom, distance-aware edge and angle features "
                    "projected to hidden width 512"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 44,
            },
            {
                "title": "Angular message passing",
                "body": (
                    "Three ALIGNN layers perform angle → edge → atom updates"
                ),
                "edge": "orange",
                "face": "light_orange",
                "wrap": 44,
            },
        ],
        "output": [
            {
                "title": "Global mean pooling",
                "body": "Mean-pool contextual atom states into one ligand vector",
                "edge": "teal",
                "face": "light_teal",
                "wrap": 25,
            },
            {
                "title": "Regression head",
                "body": "Linear 512 → 1; predict normalised pKD",
                "edge": "blue",
                "face": "light_blue",
                "wrap": 25,
            },
            {
                "title": "Inverse transform",
                "body": "pKD = z·σtrain + μtrain",
                "edge": "green",
                "face": "light_green",
                "wrap": 25,
            },
        ],
        "footer_title": "Controlled experimental role",
        "footer_body": (
            "8,118,529 trainable parameters. Comparison with the 3D distance "
            "GNN isolates the additional value of line-graph angular processing."
        ),
        "filename": "appendix_architecture_3d_alignn",
    },

    "masked_alignn": {
        "title": "Masked-pretrained crystallographic ALIGNN",
        "panel_a": "Local graph and angular representation",
        "panel_b": "Pretrained ALIGNN encoder",
        "panel_c": "Ligand-level regression",
        "input_mode": "parallel",
        "inputs": [
            {
                "title": "Atom and bond chemistry",
                "body": (
                    "152-dimensional atom features and "
                    "12-dimensional chemical-bond features"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 35,
            },
            {
                "title": "Crystal spatial graph",
                "body": (
                    "Chemical bonds plus spatial neighbours ≤5 Å; "
                    "40-bin interatomic-distance RBF"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 37,
            },
            {
                "title": "Line graph",
                "body": (
                    "Connected local edges define angles; "
                    "angle cosines expanded into 40 RBF bins"
                ),
                "edge": "orange",
                "face": "light_orange",
                "wrap": 36,
            },
        ],
        "encoder": [
            {
                "title": "Feature embedding",
                "body": (
                    "Atom, distance-aware edge and angle features "
                    "projected to hidden width 512"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 44,
            },
            {
                "title": "Angular message passing",
                "body": (
                    "Three ALIGNN layers perform angle → edge → atom updates"
                ),
                "edge": "orange",
                "face": "light_orange",
                "wrap": 44,
            },
        ],
        "output": [
            {
                "title": "Global mean pooling",
                "body": "Mean-pool contextual atom states into one ligand vector",
                "edge": "teal",
                "face": "light_teal",
                "wrap": 25,
            },
            {
                "title": "Regression head",
                "body": "Linear 512 → 1; predict normalised pKD",
                "edge": "blue",
                "face": "light_blue",
                "wrap": 25,
            },
            {
                "title": "Inverse transform",
                "body": "pKD = z·σtrain + μtrain",
                "edge": "green",
                "face": "light_green",
                "wrap": 25,
            },
        ],
        "masked": True,
        "footer_title": "Training-fold-only masked atom-feature pretraining",
        "footer_body": (
            "Zero ≈20% of complete 152-D atom vectors → ALIGNN encoder → "
            "temporary 512→152 decoder → masked-node MSE for 30 epochs. "
            "Discard the decoder and fine-tune all encoder weights. "
            "Topology, distances and angles remain visible."
        ),
        "filename": "appendix_architecture_masked_alignn",
    },

    "masked_mgt": {
        "title": "Masked-pretrained ligand Molecular Graph Transformer",
        "panel_a": "Input features and three graph views",
        "panel_b": "Pretrained complete MGT encoder",
        "panel_c": "Ligand-level regression",
        "input_mode": "parallel",
        "inputs": [
            {
                "title": "Atom state",
                "body": (
                    "90-dimensional elemental vector plus deterministic "
                    "10-dimensional Laplacian positional encoding"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 36,
            },
            {
                "title": "Local graph",
                "body": (
                    "Neighbour radius ≤8 Å; maximum 12 neighbours; "
                    "distance expanded into 80 RBF bins"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 36,
            },
            {
                "title": "Line graph",
                "body": (
                    "Local edges become nodes; angle cosine in [−1, 1] "
                    "expanded into 40 RBF bins"
                ),
                "edge": "orange",
                "face": "light_orange",
                "wrap": 36,
            },
            {
                "title": "Wider graph",
                "body": (
                    "All non-self atom pairs; Coulomb interaction "
                    "Cᵢⱼ = ZᵢZⱼ/rᵢⱼ"
                ),
                "edge": "purple",
                "face": "light_purple",
                "wrap": 36,
            },
        ],
        "encoder": [
            {
                "title": "Embedded graph tensors",
                "body": (
                    "Atom, local-edge, line-graph and wider-edge features "
                    "projected to hidden width 512"
                ),
                "edge": "navy",
                "face": "light_grey",
                "wrap": 46,
            },
            {
                "title": "Wider-graph attention",
                "body": (
                    "One multi-head layer; four attention heads; dropout 0.2"
                ),
                "edge": "purple",
                "face": "light_purple",
                "wrap": 46,
            },
            {
                "title": "Angular message passing",
                "body": (
                    "Three ALIGNN layers perform angle → edge → atom updates"
                ),
                "edge": "orange",
                "face": "light_orange",
                "wrap": 46,
            },
            {
                "title": "Local refinement",
                "body": (
                    "Three post-ALIGNN EdgeGatedGraphConv layers"
                ),
                "edge": "green",
                "face": "light_green",
                "wrap": 46,
            },
            {
                "title": "Atom-wise output block",
                "body": (
                    "512 → 512 → 512 feed-forward block; "
                    "LayerNorm and additive residual"
                ),
                "edge": "blue",
                "face": "light_blue",
                "wrap": 46,
            },
        ],
        "output": [
            {
                "title": "Global mean pooling",
                "body": "Mean-pool contextual atom states into one ligand vector",
                "edge": "teal",
                "face": "light_teal",
                "wrap": 25,
            },
            {
                "title": "Regression head",
                "body": "Linear 512 → 1; predict normalised pKD",
                "edge": "blue",
                "face": "light_blue",
                "wrap": 25,
            },
            {
                "title": "Inverse transform",
                "body": "pKD = z·σtrain + μtrain",
                "edge": "green",
                "face": "light_green",
                "wrap": 25,
            },
        ],
        "masked": True,
        "footer_title": "Training-fold-only masked atom-feature pretraining",
        "footer_body": (
            "Zero ≈20% of complete 90-D elemental atom vectors → MGT encoder → "
            "temporary 512→90 decoder → masked-node MSE for 30 epochs. "
            "Discard the decoder and fine-tune all encoder weights. "
            "Topology, distances, angles and Coulomb edges remain visible."
        ),
        "filename": "appendix_architecture_masked_mgt",
    },
}


# =========================================================================
# ARCHITECTURE FIGURE GENERATOR
# =========================================================================

# Apply the same layout and export procedure to one model specification.
def draw_appendix_architecture(configuration):
    """Draw and save one aligned appendix architecture figure."""

    fig, ax = canvas(
        (18, 10),
        configuration["title"],
    )

    # ---------------------------------------------------------------------
    # Panel headings.
    # ---------------------------------------------------------------------

    panel_label(
        ax,
        0.035,
        0.92,
        "A",
        configuration["panel_a"],
    )

    panel_label(
        ax,
        0.375,
        0.92,
        "B",
        configuration["panel_b"],
    )

    panel_label(
        ax,
        0.805,
        0.92,
        "C",
        configuration["panel_c"],
    )

    # ---------------------------------------------------------------------
    # Draw panel A.
    # ---------------------------------------------------------------------

    input_y, input_height = stack_positions(
        len(configuration["inputs"])
    )

    input_boxes = draw_architecture_stack(
        ax=ax,
        specifications=configuration["inputs"],
        x=PANEL_A_X,
        width=PANEL_A_WIDTH,
        positions=input_y,
        height=input_height,
    )

    # ---------------------------------------------------------------------
    # Draw panel B.
    # ---------------------------------------------------------------------

    encoder_y, encoder_height = stack_positions(
        len(configuration["encoder"])
    )

    encoder_boxes = draw_architecture_stack(
        ax=ax,
        specifications=configuration["encoder"],
        x=PANEL_B_X,
        width=PANEL_B_WIDTH,
        positions=encoder_y,
        height=encoder_height,
    )

    connect_vertical_stack(
        ax,
        encoder_boxes,
    )

    # ---------------------------------------------------------------------
    # Connect panel A to panel B.
    # ---------------------------------------------------------------------

    if configuration["input_mode"] == "parallel":

        connect_parallel_inputs(
            ax,
            input_boxes,
            encoder_boxes[0],
        )

    elif configuration["input_mode"] == "sequence":

        connect_sequential_inputs(
            ax,
            input_boxes,
            encoder_boxes[0],
        )

    else:
        raise ValueError(
            f"Unknown input mode: {configuration['input_mode']}"
        )

    # ---------------------------------------------------------------------
    # Draw panel C.
    # ---------------------------------------------------------------------

    output_y, output_height = output_positions(
        len(configuration["output"])
    )

    output_boxes = draw_architecture_stack(
        ax=ax,
        specifications=configuration["output"],
        x=PANEL_C_X,
        width=PANEL_C_WIDTH,
        positions=output_y,
        height=output_height,
    )

    connect_vertical_stack(
        ax,
        output_boxes,
    )

    # ---------------------------------------------------------------------
    # Connect the encoder to pooling/regression.
    # ---------------------------------------------------------------------

    connect_encoder_to_output(
        ax,
        encoder_boxes[-1],
        output_boxes[0],
    )

    # ---------------------------------------------------------------------
    # Draw the explanatory appendix banner.
    # ---------------------------------------------------------------------

    footer_face = (
        PALETTE["light_red"]
        if configuration.get("masked", False)
        else "#f7f8fa"
    )

    footer_edge = (
        PALETTE["red"]
        if configuration.get("masked", False)
        else PALETTE["navy"]
    )

    box(
        ax,
        0.045,
        0.045,
        0.685,
        0.115,
        configuration["footer_title"],
        configuration["footer_body"],
        face=footer_face,
        edge=footer_edge,
        wrap=90,
        title_size=11.5,
        body_size=9.3,
        lw=1.5,
    )

    # ---------------------------------------------------------------------
    # Save and display.
    # ---------------------------------------------------------------------

    fig.tight_layout(
        rect=(0.01, 0.01, 0.99, 0.95),
    )

    save_figure(
        fig,
        configuration["filename"],
    )

    plt.show()

    return fig


# =========================================================================
# GENERATE ALL APPENDIX FIGURES
# =========================================================================

# Export models in a fixed order so filenames and appendix placement stay reproducible.
appendix_model_order = [
    "morgan_mlp",
    "2d_gnn",
    "3d_distance_gnn",
    "3d_alignn",
    "masked_alignn",
    "masked_mgt",
]

for model_key in appendix_model_order:

    print(
        f"Creating {ARCHITECTURE_CONFIGURATIONS[model_key]['filename']}"
    )

    draw_appendix_architecture(
        ARCHITECTURE_CONFIGURATIONS[model_key]
    )

## Dissertation placement and selection

| Figure | Main purpose | Recommended location | Priority |
|---|---|---|---|
| 2.1 Overall workflow | Orient the reader to the full experiment | Section 2.1 | Essential |
| 2.2 Curation and geometry | Document selection and prove crystal-coordinate preservation | Sections 2.2–2.4 | Main text or appendix |
| 2.3 Five-fold CV | Explain outer testing, inner validation and leakage prevention | Sections 2.5–2.7 | Essential |
| 2.4 Model hierarchy | State the controlled ablation logic | Section 2.8 | Essential |
| 2.5 Adapted MGT | Explain the three graph views and complete encoder | Sections 2.9–2.11 | Essential |
| 2.6 Training and masking | Distinguish supervised fitting, pretraining and final evaluation | Sections 2.12–2.17 | Main text or appendix |

Avoid repeating performance plots in the methodology chapter. Parameter-count-versus-RMSE, learning curves and prediction-error figures belong in Results. Equations for pKD, target normalisation, Coulomb edges, angle cosines and Huber loss should remain in the text beside the relevant figure rather than becoming separate decorative plots.

### Code used: Saved-figure manifest

I collect the files registered by `save_figure` into a CSV manifest. The compound-level 3D panel saves its PNG/PDF separately and is not included in this list.


In [ ]:
# Audit only exports registered by the shared save_figure helper.
manifest = pd.DataFrame(ARTIFACTS)
manifest_path = OUTPUT_ROOT / 'methodology_figure_manifest.csv'
manifest.to_csv(manifest_path, index=False)
display(manifest)
print(f'Saved {manifest.figure.nunique()} figures in PNG, PDF and SVG formats.')
print(f'Manifest: {manifest_path}')